# MOL_LAB DIDACTIC v0.4.0

**Molekülwerkstatt für den Chemieunterricht der SEK2 – mit kuratierten Vergleichspaaren**

Ziel: Von der Molekülstruktur zu Stoffeigenschaften schließen.

Didaktischer Ablauf: **Beobachten → Vermuten → Antwort prüfen → Erklärung anzeigen**.

Diese Version nutzt bewusst nur schnelle, robuste Open-Source-Bausteine: RDKit, py3Dmol, ipywidgets und pandas. xTB/CREST bleiben im großen MOL_LAB.

> **v0.2:** kompakteres Layout mit Eigenschaften rechts neben 2D/3D, kompaktere Isomeren-Vergleichskarten und zusätzliche Sicherheit zum Neuaufbau der 3D-Anzeige.


> **v0.2:** alternative schlanke 3Dmol.js-Engine ergänzt, um Colab-Rendering-Probleme nach vielen 3D-Aufrufen abzufedern.


> **v0.4.0:** Modul *Polarität und Löslichkeit* enthält nun kuratierte Vergleichspaare mit Mini-Aufgaben und optionalem experimentellem Bezug. Die 3D-Darstellung wurde auf vier direkte Schaltflächen umgestellt.

## 1. Installation

In Google Colab: diese Zelle einmal ausführen. Die Installation dauert normalerweise unter einer Minute.


In [ ]:
# @title 🔧 Installation ausführen { display-mode: "form" }
!pip -q install rdkit py3Dmol ipywidgets pandas


## 2. Datenbank und Funktionen laden

Diese Zelle startet die eigentliche Oberfläche. In Colab ist sie als **Form-Zelle** vorbereitet; bei Bedarf kann über **Code anzeigen/ausblenden** zwischen Bedienoberfläche und technischem Code umgeschaltet werden.


In [ ]:
# @title 🧪 MOL_LAB DIDACTIC starten { display-mode: "form" }
import json, math, textwrap, html, base64, uuid, gc
from io import BytesIO
import pandas as pd
from IPython.display import display, HTML, clear_output, Javascript
import ipywidgets as widgets
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, Descriptors, rdMolDescriptors, Crippen
from rdkit.Chem.Draw import IPythonConsole
import py3Dmol


DATA_URL = "https://raw.githubusercontent.com/ekerzendorfer/MOL_LAB_DIDACTIC/main/molecules_v0_4_0.json"
LOCAL_DATA_FILE = "molecules_v0_4_0.json"

# Fallback-Daten: werden verwendet, wenn die externe JSON-Datei in Colab noch nicht erreichbar ist.
FALLBACK_DATA = {'meta': {'schema_version': '0.4.0', 'title': 'MOL_LAB DIDACTIC Unterrichtsdaten', 'language': 'de', 'description': 'Strukturierte Moleküldaten mit didaktischen Karten, Stoffdaten, Reaktivitäts-Hinweisen und kuratierten Vergleichspaaren.', 'notes': ['Die App berechnet Summenformel, molare Masse, logP, TPSA und weitere Deskriptoren weiterhin aus dem SMILES.', 'Stoffdaten sind kuratierte Unterrichtswerte und werden nur dort hinterlegt, wo sie didaktisch nützlich sind.', 'Reaktivitäts-Hinweise sind qualitative Orientierung anhand funktioneller Gruppen und keine vollständige Reaktionsvorhersage.', 'Alternative JSON-Datenpakete können später dieselbe Struktur nutzen.', 'v0.4.0 macht Reaktivitäts-Hinweise in der Oberfläche sichtbar, wo sie didaktisch sinnvoll sind.', 'SchülerInnen-Protokolle werden aus Leitfragen, Modellwerten, Mini-Aufgaben und optionalen Stoffdaten erzeugt.', 'Stoffdaten werden vor allem in Molekül- und Isomerenvergleichen angezeigt, damit sie als Struktur-Eigenschaft-Bezug verstanden werden.']}, 'modules': {'Polarität und Löslichkeit': [{'name': 'Wasser', 'smiles': 'O', 'niveau': 'Basis', 'leitfrage': 'Warum ist Wasser ein stark polares Lösungsmittel?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage erklärt die besonderen Stoffeigenschaften von Wasser am besten?', 'antworten': ['Die gewinkelte Struktur und O-H-Bindungen ermöglichen ein Netzwerk aus Wasserstoffbrücken.', 'Wasser ist nur deshalb flüssig, weil seine Moleküle besonders schwer sind.', 'Wasser ist unpolar, aber die Moleküle ziehen sich durch Ionenbindungen an.', 'Die Summenformel H2O reicht allein aus, um Siedepunkt und Löslichkeit zu erklären.'], 'richtig': 0, 'feedback': 'Richtig: Die Kombination aus polarer O-H-Bindung, gewinkelter Struktur und Wasserstoffbrücken ist entscheidend.', 'erklaerung': 'Wasser besitzt polare O-H-Bindungen und eine gewinkelte Struktur; dadurch heben sich die Dipole nicht auf.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'wasser', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Wasser ein stark polares Lösungsmittel?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage erklärt die besonderen Stoffeigenschaften von Wasser am besten?', 'mc_antworten': ['Die gewinkelte Struktur und O-H-Bindungen ermöglichen ein Netzwerk aus Wasserstoffbrücken.', 'Wasser ist nur deshalb flüssig, weil seine Moleküle besonders schwer sind.', 'Wasser ist unpolar, aber die Moleküle ziehen sich durch Ionenbindungen an.', 'Die Summenformel H2O reicht allein aus, um Siedepunkt und Löslichkeit zu erklären.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die Kombination aus polarer O-H-Bindung, gewinkelter Struktur und Wasserstoffbrücken ist entscheidend.', 'erklaerung': 'Wasser besitzt polare O-H-Bindungen und eine gewinkelte Struktur; dadurch heben sich die Dipole nicht auf.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '0', 'siedepunkt_C': '100', 'dichte_g_cm3': '0.998', 'wasserloeslichkeit': 'mit Wasser vollständig mischbar', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten; Werte je nach Quelle/Temperatur leicht abweichend', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv als Lösungsmittel und Reaktionspartner', 'warum': 'Wasser kann Protonen übertragen und an Hydrolysen beteiligt sein.', 'reaktionsidee': 'Säure-Base-Reaktionen, Hydratation, Hydrolyse.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Methan', 'smiles': 'C', 'niveau': 'Basis', 'leitfrage': 'Warum ist Methan unpolar und schlecht wasserlöslich?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage beschreibt Methan im Vergleich zu Wasser am besten?', 'antworten': ['Methan ist nahezu unpolar und besitzt keine geeigneten Gruppen für Wasserstoffbrücken.', 'Methan ist wegen seiner tetraedrischen Form stark polar.', 'Methan ist wasserlöslich, weil es sehr klein ist.', 'Methan enthält zwar keine Heteroatome, kann aber starke H-Brücken bilden.'], 'richtig': 0, 'feedback': 'Richtig: Methan ist nahezu unpolar; es wechselwirkt vor allem über London-Kräfte.', 'erklaerung': 'Methan ist sehr symmetrisch und besitzt keine stark polare funktionelle Gruppe.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'methan', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Methan unpolar und schlecht wasserlöslich?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage beschreibt Methan im Vergleich zu Wasser am besten?', 'mc_antworten': ['Methan ist nahezu unpolar und besitzt keine geeigneten Gruppen für Wasserstoffbrücken.', 'Methan ist wegen seiner tetraedrischen Form stark polar.', 'Methan ist wasserlöslich, weil es sehr klein ist.', 'Methan enthält zwar keine Heteroatome, kann aber starke H-Brücken bilden.'], 'mc_richtig': 0, 'feedback': 'Richtig: Methan ist nahezu unpolar; es wechselwirkt vor allem über London-Kräfte.', 'erklaerung': 'Methan ist sehr symmetrisch und besitzt keine stark polare funktionelle Gruppe.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−182.5', 'siedepunkt_C': '−161.5', 'dichte_g_cm3': None, 'wasserloeslichkeit': 'kaum wasserlöslich', 'aggregatzustand_raumtemperatur': 'gasförmig', 'bedingungen': '1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'eher reaktionsträge', 'warum': 'Methan besitzt nur C–H-Bindungen und keine stark polare funktionelle Gruppe.', 'reaktionsidee': 'Verbrennung nach Aktivierung; radikalische Halogenierung unter geeigneten Bedingungen.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Ammoniak', 'smiles': 'N', 'niveau': 'Basis', 'leitfrage': 'Warum ist Ammoniak polar und gut wasserlöslich?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten zu Ammoniak?', 'antworten': ['Ammoniak ist polar und kann über N-H-Bindungen Wasserstoffbrücken bilden.', 'Ammoniak ist linear und daher unpolar wie CO2.', 'Ammoniak ist nur wegen seiner geringen molaren Masse gut wasserlöslich.', 'Ammoniak enthält Stickstoff und ist deshalb automatisch ein Salz.'], 'richtig': 0, 'feedback': 'Richtig: Ammoniak ist trigonal-pyramidal, polar und kann H-Brücken bilden.', 'erklaerung': 'Ammoniak besitzt polare N-H-Bindungen und ein freies Elektronenpaar am Stickstoff.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'ammoniak', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Ammoniak polar und gut wasserlöslich?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten zu Ammoniak?', 'mc_antworten': ['Ammoniak ist polar und kann über N-H-Bindungen Wasserstoffbrücken bilden.', 'Ammoniak ist linear und daher unpolar wie CO2.', 'Ammoniak ist nur wegen seiner geringen molaren Masse gut wasserlöslich.', 'Ammoniak enthält Stickstoff und ist deshalb automatisch ein Salz.'], 'mc_richtig': 0, 'feedback': 'Richtig: Ammoniak ist trigonal-pyramidal, polar und kann H-Brücken bilden.', 'erklaerung': 'Ammoniak besitzt polare N-H-Bindungen und ein freies Elektronenpaar am Stickstoff.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−77.7', 'siedepunkt_C': '−33.3', 'dichte_g_cm3': None, 'wasserloeslichkeit': 'sehr gut löslich; bildet basische Lösungen', 'aggregatzustand_raumtemperatur': 'gasförmig', 'bedingungen': '1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv als Base und Nukleophil', 'warum': 'Das freie Elektronenpaar am Stickstoff kann Protonen aufnehmen oder Bindungen bilden.', 'reaktionsidee': 'Bildung von Ammoniumsalzen; Reaktionen als Nukleophil.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Kohlenstoffdioxid', 'smiles': 'O=C=O', 'niveau': 'Basis', 'leitfrage': 'Warum ist CO₂ trotz polarer C=O-Bindungen insgesamt kaum polar?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist CO2 trotz polarer C=O-Bindungen insgesamt kaum polar?', 'antworten': ['Die lineare, symmetrische Anordnung lässt die Bindungsdipole einander aufheben.', 'C=O-Bindungen sind grundsätzlich unpolar.', 'CO2 enthält Sauerstoff und muss daher stark polar sein.', 'CO2 ist unpolar, weil es keine Doppelbindungen enthält.'], 'richtig': 0, 'feedback': 'Richtig: Die Molekülgeometrie entscheidet mit; bei CO2 heben sich die Bindungsdipole auf.', 'erklaerung': 'CO₂ besitzt polare Bindungen, aber die lineare Anordnung führt zur Aufhebung der Dipole.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'kohlenstoffdioxid', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist CO₂ trotz polarer C=O-Bindungen insgesamt kaum polar?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist CO2 trotz polarer C=O-Bindungen insgesamt kaum polar?', 'mc_antworten': ['Die lineare, symmetrische Anordnung lässt die Bindungsdipole einander aufheben.', 'C=O-Bindungen sind grundsätzlich unpolar.', 'CO2 enthält Sauerstoff und muss daher stark polar sein.', 'CO2 ist unpolar, weil es keine Doppelbindungen enthält.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die Molekülgeometrie entscheidet mit; bei CO2 heben sich die Bindungsdipole auf.', 'erklaerung': 'CO₂ besitzt polare Bindungen, aber die lineare Anordnung führt zur Aufhebung der Dipole.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': 'sublimiert bei −78.5', 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': 'begrenzt löslich; reagiert teilweise zu Kohlensäure/Hydrogencarbonat', 'aggregatzustand_raumtemperatur': 'gasförmig', 'bedingungen': '1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv gegenüber Basen/Nukleophilen', 'warum': 'CO2 ist linear und insgesamt unpolar, das C-Atom kann aber von starken Nukleophilen angegriffen werden.', 'reaktionsidee': 'Bildung von Carbonaten/Hydrogencarbonaten; Reaktion mit Wasser/Basen.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Methanol', 'smiles': 'CO', 'niveau': 'Basis', 'leitfrage': 'Warum ist Methanol sehr gut mit Wasser mischbar?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Methanol sehr gut mit Wasser mischbar?', 'antworten': ['Die OH-Gruppe bildet H-Brücken und der unpolare Rest ist sehr klein.', 'Methanol ist gut löslich, weil alle organischen Moleküle mit Sauerstoff vollständig wasserlöslich sind.', 'Methanol enthält eine Carbonylgruppe, die stark sauer reagiert.', 'Die C-H-Bindungen sind der Hauptgrund für die Wasserlöslichkeit.'], 'richtig': 0, 'feedback': 'Richtig: Die OH-Gruppe dominiert hier die Eigenschaften, weil der unpolare Teil klein ist.', 'erklaerung': 'Methanol besitzt eine polare OH-Gruppe und nur eine kleine Methylgruppe.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'methanol', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Methanol sehr gut mit Wasser mischbar?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Methanol sehr gut mit Wasser mischbar?', 'mc_antworten': ['Die OH-Gruppe bildet H-Brücken und der unpolare Rest ist sehr klein.', 'Methanol ist gut löslich, weil alle organischen Moleküle mit Sauerstoff vollständig wasserlöslich sind.', 'Methanol enthält eine Carbonylgruppe, die stark sauer reagiert.', 'Die C-H-Bindungen sind der Hauptgrund für die Wasserlöslichkeit.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die OH-Gruppe dominiert hier die Eigenschaften, weil der unpolare Teil klein ist.', 'erklaerung': 'Methanol besitzt eine polare OH-Gruppe und nur eine kleine Methylgruppe.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−97.6', 'siedepunkt_C': '64.7', 'dichte_g_cm3': '0.79', 'wasserloeslichkeit': 'vollständig mischbar', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Die OH-Gruppe ermöglicht Oxidation, Veresterung und Säure-Base-Wechselwirkungen.', 'reaktionsidee': 'Oxidation zu Methanal; Bildung von Estern.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Ethanol', 'smiles': 'CCO', 'niveau': 'Basis', 'leitfrage': 'Warum ist Ethanol mit Wasser mischbar, obwohl es auch einen unpolaren Teil besitzt?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Ethanol gut mit Wasser mischbar, obwohl es einen Kohlenwasserstoffrest besitzt?', 'antworten': ['Die OH-Gruppe kann H-Brücken bilden; der unpolare Rest ist noch relativ klein.', 'Der Ethylrest bildet starke H-Brücken mit Wasser.', 'Ethanol ist gut wasserlöslich, weil es keine unpolaren Bereiche besitzt.', 'Ethanol ist ionisch und löst sich deshalb wie Kochsalz.'], 'richtig': 0, 'feedback': 'Richtig: Ethanol vereint eine polare OH-Gruppe mit einem noch kleinen unpolaren Rest.', 'erklaerung': 'Ethanol besitzt eine polare OH-Gruppe und einen noch kleinen unpolaren Ethylrest.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'ethanol', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Ethanol mit Wasser mischbar, obwohl es auch einen unpolaren Teil besitzt?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Ethanol gut mit Wasser mischbar, obwohl es einen Kohlenwasserstoffrest besitzt?', 'mc_antworten': ['Die OH-Gruppe kann H-Brücken bilden; der unpolare Rest ist noch relativ klein.', 'Der Ethylrest bildet starke H-Brücken mit Wasser.', 'Ethanol ist gut wasserlöslich, weil es keine unpolaren Bereiche besitzt.', 'Ethanol ist ionisch und löst sich deshalb wie Kochsalz.'], 'mc_richtig': 0, 'feedback': 'Richtig: Ethanol vereint eine polare OH-Gruppe mit einem noch kleinen unpolaren Rest.', 'erklaerung': 'Ethanol besitzt eine polare OH-Gruppe und einen noch kleinen unpolaren Ethylrest.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−114.1', 'siedepunkt_C': '78.4', 'dichte_g_cm3': '0.789', 'wasserloeslichkeit': 'vollständig mischbar', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Die OH-Gruppe ermöglicht Oxidation, Veresterung und Wasserstoffbrücken.', 'reaktionsidee': 'Oxidation zu Ethanal/Essigsäure; Esterbildung.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'n-Butanol', 'smiles': 'CCCCO', 'niveau': 'Basis', 'leitfrage': 'Warum ist n-Butanol schlechter wasserlöslich als Methanol oder Ethanol?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist n-Butanol weniger wasserfreundlich als Methanol oder Ethanol?', 'antworten': ['Die OH-Gruppe bleibt polar, aber der größere unpolare Rest wird stärker wirksam.', 'n-Butanol besitzt keine OH-Gruppe mehr.', 'Die längere Kohlenstoffkette macht die O-H-Bindung ionisch.', 'Alle Alkohole sind unabhängig von der Kettenlänge gleich gut wasserlöslich.'], 'richtig': 0, 'feedback': 'Richtig: Mit wachsender Alkylkette nimmt der unpolare Anteil zu.', 'erklaerung': 'Die OH-Gruppe bleibt polar, aber mit wachsender Alkylkette nimmt der unpolare Anteil zu.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'n_butanol', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist n-Butanol schlechter wasserlöslich als Methanol oder Ethanol?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist n-Butanol weniger wasserfreundlich als Methanol oder Ethanol?', 'mc_antworten': ['Die OH-Gruppe bleibt polar, aber der größere unpolare Rest wird stärker wirksam.', 'n-Butanol besitzt keine OH-Gruppe mehr.', 'Die längere Kohlenstoffkette macht die O-H-Bindung ionisch.', 'Alle Alkohole sind unabhängig von der Kettenlänge gleich gut wasserlöslich.'], 'mc_richtig': 0, 'feedback': 'Richtig: Mit wachsender Alkylkette nimmt der unpolare Anteil zu.', 'erklaerung': 'Die OH-Gruppe bleibt polar, aber mit wachsender Alkylkette nimmt der unpolare Anteil zu.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−89.8', 'siedepunkt_C': '117.7', 'dichte_g_cm3': '0.81', 'wasserloeslichkeit': 'begrenzt löslich', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Die OH-Gruppe reagiert wie bei Alkoholen, der unpolare Rest beeinflusst vor allem Stoffeigenschaften.', 'reaktionsidee': 'Oxidation und Esterbildung.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Methanal', 'smiles': 'C=O', 'niveau': 'Basis', 'leitfrage': 'Warum ist Methanal polar, obwohl es keine OH-Gruppe besitzt?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Was unterscheidet Methanal als Carbonylverbindung von Methanol?', 'antworten': ['Methanal kann H-Brücken aufnehmen, aber keine O-H-H-Brücken spenden.', 'Methanal besitzt wie Methanol eine Alkoholgruppe.', 'Methanal ist völlig unpolar, weil es nur ein O-Atom enthält.', 'Methanal und Methanol haben dieselbe funktionelle Gruppe.'], 'richtig': 0, 'feedback': 'Richtig: Methanal enthält eine Carbonylgruppe, keine OH-Gruppe.', 'erklaerung': 'Die C=O-Doppelbindung ist polar; der Sauerstoff kann als Akzeptor wirken.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'methanal', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Methanal polar, obwohl es keine OH-Gruppe besitzt?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Was unterscheidet Methanal als Carbonylverbindung von Methanol?', 'mc_antworten': ['Methanal kann H-Brücken aufnehmen, aber keine O-H-H-Brücken spenden.', 'Methanal besitzt wie Methanol eine Alkoholgruppe.', 'Methanal ist völlig unpolar, weil es nur ein O-Atom enthält.', 'Methanal und Methanol haben dieselbe funktionelle Gruppe.'], 'mc_richtig': 0, 'feedback': 'Richtig: Methanal enthält eine Carbonylgruppe, keine OH-Gruppe.', 'erklaerung': 'Die C=O-Doppelbindung ist polar; der Sauerstoff kann als Akzeptor wirken.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−92', 'siedepunkt_C': '−19', 'dichte_g_cm3': None, 'wasserloeslichkeit': 'gut wasserlöslich; in Wasser chemisch nicht unverändert (Hydratbildung/Polymerisation möglich)', 'aggregatzustand_raumtemperatur': 'gasförmig', 'bedingungen': '1 atm; Formaldehyd wird im Labor oft als wässrige Lösung betrachtet', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv', 'warum': 'Die Carbonylgruppe eines Aldehyds ist elektrophil; Methanal ist besonders reaktiv.', 'reaktionsidee': 'Nukleophile Addition; Oxidation zu Ameisensäure; Hydratbildung.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Aceton', 'smiles': 'CC(=O)C', 'niveau': 'Basis', 'leitfrage': 'Warum ist Aceton gut mit Wasser mischbar, obwohl es keine OH-Gruppe besitzt?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage beschreibt Aceton als Lösungsmittel am besten?', 'antworten': ['Aceton ist polar aprotisch: es kann H-Brücken aufnehmen, aber keine spenden.', 'Aceton ist ein Alkohol und kann deshalb H-Brücken spenden.', 'Aceton ist unpolar, weil es drei Kohlenstoffatome enthält.', 'Aceton besitzt eine Carboxylgruppe und reagiert sauer.'], 'richtig': 0, 'feedback': 'Richtig: Die C=O-Gruppe macht Aceton polar, aber ohne O-H/N-H kann es keine H-Brücken spenden.', 'erklaerung': 'Aceton besitzt eine polare Carbonylgruppe und zwei kleine unpolare Methylgruppen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'aceton', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Aceton gut mit Wasser mischbar, obwohl es keine OH-Gruppe besitzt?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage beschreibt Aceton als Lösungsmittel am besten?', 'mc_antworten': ['Aceton ist polar aprotisch: es kann H-Brücken aufnehmen, aber keine spenden.', 'Aceton ist ein Alkohol und kann deshalb H-Brücken spenden.', 'Aceton ist unpolar, weil es drei Kohlenstoffatome enthält.', 'Aceton besitzt eine Carboxylgruppe und reagiert sauer.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die C=O-Gruppe macht Aceton polar, aber ohne O-H/N-H kann es keine H-Brücken spenden.', 'erklaerung': 'Aceton besitzt eine polare Carbonylgruppe und zwei kleine unpolare Methylgruppen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−95', 'siedepunkt_C': '56.1', 'dichte_g_cm3': '0.79', 'wasserloeslichkeit': 'vollständig mischbar', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Die Carbonylgruppe ist elektrophil, aber Ketone sind gegenüber Oxidation stabiler als Aldehyde.', 'reaktionsidee': 'Nukleophile Addition an die C=O-Gruppe.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Diethylether', 'smiles': 'CCOCC', 'niveau': 'Basis', 'leitfrage': 'Warum unterscheidet sich Diethylether deutlich von Alkoholen?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Diethylether deutlich anders als Ethanol?', 'antworten': ['Er kann H-Brücken aufnehmen, aber mangels O-H-Gruppe nicht spenden.', 'Er enthält keine polaren Bindungen.', 'Er ist wegen des Sauerstoffatoms automatisch vollständig mit Wasser mischbar.', 'Er besitzt eine Carboxylgruppe und reagiert sauer.'], 'richtig': 0, 'feedback': 'Richtig: Ether-Sauerstoff kann H-Brücken aufnehmen, aber Diethylether kann keine spenden.', 'erklaerung': 'Diethylether enthält ein O-Atom, aber keine O-H-Bindung.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'diethylether', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum unterscheidet sich Diethylether deutlich von Alkoholen?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Diethylether deutlich anders als Ethanol?', 'mc_antworten': ['Er kann H-Brücken aufnehmen, aber mangels O-H-Gruppe nicht spenden.', 'Er enthält keine polaren Bindungen.', 'Er ist wegen des Sauerstoffatoms automatisch vollständig mit Wasser mischbar.', 'Er besitzt eine Carboxylgruppe und reagiert sauer.'], 'mc_richtig': 0, 'feedback': 'Richtig: Ether-Sauerstoff kann H-Brücken aufnehmen, aber Diethylether kann keine spenden.', 'erklaerung': 'Diethylether enthält ein O-Atom, aber keine O-H-Bindung.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−116', 'siedepunkt_C': '34.6', 'dichte_g_cm3': '0.71', 'wasserloeslichkeit': 'begrenzt löslich', 'aggregatzustand_raumtemperatur': 'flüssig, sehr flüchtig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'relativ reaktionsträge', 'warum': 'Ether besitzen keine O–H-Bindung und reagieren unter milden Bedingungen meist weniger stark als Alkohole.', 'reaktionsidee': 'Spaltung unter starken sauren Bedingungen; Peroxidbildung als Sicherheitsaspekt.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Essigsäure', 'smiles': 'CC(=O)O', 'niveau': 'Basis', 'leitfrage': 'Warum ist Essigsäure stark polar und gut wasserlöslich?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Essigsäure besonders gut für Struktur-Eigenschafts-Vergleiche geeignet?', 'antworten': ['Die Carboxylgruppe verbindet Polarität, H-Brücken und Säure-Base-Verhalten.', 'Essigsäure ist ein völlig unpolares Molekül.', 'Essigsäure enthält nur eine Ethergruppe.', 'Die Methylgruppe allein erklärt die hohe Wasserlöslichkeit.'], 'richtig': 0, 'feedback': 'Richtig: Die Carboxylgruppe ist der zentrale Strukturbaustein für die Eigenschaften.', 'erklaerung': 'Essigsäure besitzt eine Carboxylgruppe mit C=O und O-H am selben Kohlenstoffatom.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'essigsaeure', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Essigsäure stark polar und gut wasserlöslich?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Essigsäure besonders gut für Struktur-Eigenschafts-Vergleiche geeignet?', 'mc_antworten': ['Die Carboxylgruppe verbindet Polarität, H-Brücken und Säure-Base-Verhalten.', 'Essigsäure ist ein völlig unpolares Molekül.', 'Essigsäure enthält nur eine Ethergruppe.', 'Die Methylgruppe allein erklärt die hohe Wasserlöslichkeit.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die Carboxylgruppe ist der zentrale Strukturbaustein für die Eigenschaften.', 'erklaerung': 'Essigsäure besitzt eine Carboxylgruppe mit C=O und O-H am selben Kohlenstoffatom.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '16.6', 'siedepunkt_C': '118.1', 'dichte_g_cm3': '1.05', 'wasserloeslichkeit': 'vollständig mischbar', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv in Säure-Base-Reaktionen', 'warum': 'Die Carboxylgruppe kann ein Proton abgeben.', 'reaktionsidee': 'Neutralisation mit Basen; Esterbildung mit Alkoholen.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Schwefelwasserstoff', 'smiles': 'S', 'niveau': 'Vergleich', 'leitfrage': 'Warum ist Schwefelwasserstoff bei Raumtemperatur gasförmig, Wasser aber flüssig?', 'beobachten': ['Vergleiche die Molekülform mit Wasser.', 'Achte darauf, ob starke Wasserstoffbrücken möglich sind.', 'Nutze die Modellwerte nur als grobe Orientierung.'], 'frage': 'Warum verhält sich H2S nicht einfach wie „schweres Wasser“?', 'antworten': ['S-H-Bindungen führen nicht zu einem wasserähnlich starken H-Brücken-Netzwerk.', 'H2S ist linear und besitzt keine freien Elektronenpaare.', 'H2S ist ein Ionengitter und daher fest.', 'H2S ist mit Wasser identisch, weil beide dreiatomig sind.'], 'richtig': 0, 'feedback': 'Richtig: H2S ist zwar gewinkelt, bildet aber keine ähnlich starken H-Brücken wie Wasser.', 'erklaerung': 'Wasser und Schwefelwasserstoff wirken auf den ersten Blick ähnlich, unterscheiden sich aber stark in ihren zwischenmolekularen Kräften. Wasser bildet ein starkes Wasserstoffbrücken-Netzwerk; bei H₂S sind diese Wechselwirkungen viel schwächer.', 'modellgrenze': 'Für sehr kleine anorganische Moleküle sind RDKit-Deskriptoren wie logP/TPSA nur eingeschränkt aussagekräftig. Hier ist der experimentelle Bezug besonders wichtig.', 'id': 'schwefelwasserstoff', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Schwefelwasserstoff bei Raumtemperatur gasförmig, Wasser aber flüssig?', 'beobachten': ['Vergleiche die Molekülform mit Wasser.', 'Achte darauf, ob starke Wasserstoffbrücken möglich sind.', 'Nutze die Modellwerte nur als grobe Orientierung.'], 'mc_frage': 'Warum verhält sich H2S nicht einfach wie „schweres Wasser“?', 'mc_antworten': ['S-H-Bindungen führen nicht zu einem wasserähnlich starken H-Brücken-Netzwerk.', 'H2S ist linear und besitzt keine freien Elektronenpaare.', 'H2S ist ein Ionengitter und daher fest.', 'H2S ist mit Wasser identisch, weil beide dreiatomig sind.'], 'mc_richtig': 0, 'feedback': 'Richtig: H2S ist zwar gewinkelt, bildet aber keine ähnlich starken H-Brücken wie Wasser.', 'erklaerung': 'Wasser und Schwefelwasserstoff wirken auf den ersten Blick ähnlich, unterscheiden sich aber stark in ihren zwischenmolekularen Kräften. Wasser bildet ein starkes Wasserstoffbrücken-Netzwerk; bei H₂S sind diese Wechselwirkungen viel schwächer.', 'modellgrenze': 'Für sehr kleine anorganische Moleküle sind RDKit-Deskriptoren wie logP/TPSA nur eingeschränkt aussagekräftig. Hier ist der experimentelle Bezug besonders wichtig.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−85.5', 'siedepunkt_C': '−60.3', 'dichte_g_cm3': None, 'wasserloeslichkeit': 'mäßig löslich; bildet keine wasserähnlichen H-Brücken-Netzwerke', 'aggregatzustand_raumtemperatur': 'gasförmig', 'bedingungen': '1 atm', 'quelle': 'kuratierte Unterrichtsdaten; Sicherheitsaspekt: giftiges Gas', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv in Redox- und Säure-Base-Zusammenhängen', 'warum': 'H2S ist eine schwache Säure und kann als Reduktionsmittel wirken.', 'reaktionsidee': 'Bildung von Sulfiden; Oxidation zu Schwefel/Schwefelverbindungen.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Hexan', 'smiles': 'CCCCCC', 'niveau': 'Vergleich', 'leitfrage': 'Warum ist Hexan flüssig, Methan aber gasförmig, obwohl beide unpolar sind?', 'beobachten': ['Vergleiche die Molekülgröße.', 'Betrachte die unpolare Oberfläche.', 'Überlege, warum größere unpolare Moleküle stärkere London-Kräfte besitzen.'], 'frage': 'Warum ist Hexan flüssig, Methan aber gasförmig?', 'antworten': ['Hexan ist größer; seine Elektronenhülle ist leichter polarisierbar, London-Kräfte sind stärker.', 'Hexan ist polarer, weil es mehr C-H-Bindungen besitzt.', 'Methan hat stärkere H-Brücken als Hexan.', 'Hexan ist ionisch, Methan nicht.'], 'richtig': 0, 'feedback': 'Richtig: Bei unpolaren Molekülen werden London-Kräfte mit Molekülgröße/Oberfläche stärker.', 'erklaerung': 'Unpolar bedeutet nicht wechselwirkungsfrei. Hexan hat eine deutlich größere Oberfläche als Methan; dadurch wirken stärkere London-Kräfte zwischen den Molekülen.', 'modellgrenze': 'Die App zeigt London-Kräfte nicht direkt, sondern nur indirekt über Molekülgröße und Oberfläche.', 'id': 'hexan', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Hexan flüssig, Methan aber gasförmig, obwohl beide unpolar sind?', 'beobachten': ['Vergleiche die Molekülgröße.', 'Betrachte die unpolare Oberfläche.', 'Überlege, warum größere unpolare Moleküle stärkere London-Kräfte besitzen.'], 'mc_frage': 'Warum ist Hexan flüssig, Methan aber gasförmig?', 'mc_antworten': ['Hexan ist größer; seine Elektronenhülle ist leichter polarisierbar, London-Kräfte sind stärker.', 'Hexan ist polarer, weil es mehr C-H-Bindungen besitzt.', 'Methan hat stärkere H-Brücken als Hexan.', 'Hexan ist ionisch, Methan nicht.'], 'mc_richtig': 0, 'feedback': 'Richtig: Bei unpolaren Molekülen werden London-Kräfte mit Molekülgröße/Oberfläche stärker.', 'erklaerung': 'Unpolar bedeutet nicht wechselwirkungsfrei. Hexan hat eine deutlich größere Oberfläche als Methan; dadurch wirken stärkere London-Kräfte zwischen den Molekülen.', 'modellgrenze': 'Die App zeigt London-Kräfte nicht direkt, sondern nur indirekt über Molekülgröße und Oberfläche.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−95', 'siedepunkt_C': '68.7', 'dichte_g_cm3': '0.66', 'wasserloeslichkeit': 'kaum wasserlöslich', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'eher reaktionsträge', 'warum': 'Hexan enthält nur C–C- und C–H-Bindungen.', 'reaktionsidee': 'Verbrennung; radikalische Substitution unter speziellen Bedingungen.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': '1-Hexanol', 'smiles': 'CCCCCCO', 'niveau': 'Vergleich', 'leitfrage': 'Warum ist 1-Hexanol viel schlechter wasserlöslich als Ethanol?', 'beobachten': ['Suche die OH-Gruppe.', 'Vergleiche den unpolaren Kohlenwasserstoffrest mit Ethanol.', 'Achte auf logP, TPSA und die Wasserlöslichkeits-Tendenz.'], 'frage': 'Was zeigt 1-Hexanol im Vergleich zu Ethanol besonders gut?', 'antworten': ['Eine OH-Gruppe reicht nicht immer aus, um einen großen unpolaren Rest zu überwiegen.', 'Alkohole verlieren mit längerer Kette ihre OH-Gruppe.', '1-Hexanol ist wegen der OH-Gruppe vollständig mit Wasser mischbar.', 'Die Kettenlänge hat keinen Einfluss auf Stoffeigenschaften.'], 'richtig': 0, 'feedback': 'Richtig: Das Verhältnis von polarer Gruppe zu unpolarem Rest ist entscheidend.', 'erklaerung': 'Bei kurzen Alkoholen dominiert die OH-Gruppe die Wechselwirkung mit Wasser. Bei 1-Hexanol ist der unpolare Kohlenwasserstoffrest so groß, dass die Wasserlöslichkeit deutlich sinkt.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': '1_hexanol', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist 1-Hexanol viel schlechter wasserlöslich als Ethanol?', 'beobachten': ['Suche die OH-Gruppe.', 'Vergleiche den unpolaren Kohlenwasserstoffrest mit Ethanol.', 'Achte auf logP, TPSA und die Wasserlöslichkeits-Tendenz.'], 'mc_frage': 'Was zeigt 1-Hexanol im Vergleich zu Ethanol besonders gut?', 'mc_antworten': ['Eine OH-Gruppe reicht nicht immer aus, um einen großen unpolaren Rest zu überwiegen.', 'Alkohole verlieren mit längerer Kette ihre OH-Gruppe.', '1-Hexanol ist wegen der OH-Gruppe vollständig mit Wasser mischbar.', 'Die Kettenlänge hat keinen Einfluss auf Stoffeigenschaften.'], 'mc_richtig': 0, 'feedback': 'Richtig: Das Verhältnis von polarer Gruppe zu unpolarem Rest ist entscheidend.', 'erklaerung': 'Bei kurzen Alkoholen dominiert die OH-Gruppe die Wechselwirkung mit Wasser. Bei 1-Hexanol ist der unpolare Kohlenwasserstoffrest so groß, dass die Wasserlöslichkeit deutlich sinkt.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−45', 'siedepunkt_C': '157', 'dichte_g_cm3': '0.81', 'wasserloeslichkeit': 'schlecht bis begrenzt löslich', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Die OH-Gruppe bleibt reaktionsfähig, auch wenn der große Alkylrest die Wasserlöslichkeit verringert.', 'reaktionsidee': 'Oxidation und Esterbildung.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Palmitinsäure', 'smiles': 'CCCCCCCCCCCCCCCC(=O)O', 'niveau': 'Vertiefung', 'leitfrage': 'Warum ist Palmitinsäure trotz Carboxylgruppe kaum wasserlöslich?', 'beobachten': ['Suche die Carboxylgruppe.', 'Vergleiche die Länge des unpolaren Restes mit Essigsäure.', 'Überlege, welcher Molekülteil die Stoffeigenschaft stärker prägt.'], 'frage': 'Warum ist Palmitinsäure trotz Carboxylgruppe kaum wasserlöslich?', 'antworten': ['Der sehr lange unpolare Kohlenwasserstoffrest dominiert die Stoffeigenschaft.', 'Palmitinsäure enthält keine polare funktionelle Gruppe.', 'Alle Carbonsäuren sind grundsätzlich unpolar.', 'Die Carboxylgruppe wird bei längeren Molekülen chemisch entfernt.'], 'richtig': 0, 'feedback': 'Richtig: Die Carboxylgruppe ist polar, aber der lange Alkylrest überwiegt stark.', 'erklaerung': 'Palmitinsäure besitzt zwar eine Carboxylgruppe, aber auch eine sehr lange unpolare Kohlenwasserstoffkette. Dadurch ist sie in Wasser kaum löslich.', 'modellgrenze': 'Das Modell berücksichtigt keine Kristallpackung, pH-Effekte oder die Bildung von Aggregaten/Mizellen.', 'id': 'palmitinsaeure', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Palmitinsäure trotz Carboxylgruppe kaum wasserlöslich?', 'beobachten': ['Suche die Carboxylgruppe.', 'Vergleiche die Länge des unpolaren Restes mit Essigsäure.', 'Überlege, welcher Molekülteil die Stoffeigenschaft stärker prägt.'], 'mc_frage': 'Warum ist Palmitinsäure trotz Carboxylgruppe kaum wasserlöslich?', 'mc_antworten': ['Der sehr lange unpolare Kohlenwasserstoffrest dominiert die Stoffeigenschaft.', 'Palmitinsäure enthält keine polare funktionelle Gruppe.', 'Alle Carbonsäuren sind grundsätzlich unpolar.', 'Die Carboxylgruppe wird bei längeren Molekülen chemisch entfernt.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die Carboxylgruppe ist polar, aber der lange Alkylrest überwiegt stark.', 'erklaerung': 'Palmitinsäure besitzt zwar eine Carboxylgruppe, aber auch eine sehr lange unpolare Kohlenwasserstoffkette. Dadurch ist sie in Wasser kaum löslich.', 'modellgrenze': 'Das Modell berücksichtigt keine Kristallpackung, pH-Effekte oder die Bildung von Aggregaten/Mizellen.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '63', 'siedepunkt_C': '>300 (Zersetzung/hoher Siedebereich)', 'dichte_g_cm3': None, 'wasserloeslichkeit': 'praktisch unlöslich', 'aggregatzustand_raumtemperatur': 'fest', 'bedingungen': 'ca. 20–25 °C', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv in Säure-Base- und Esterbildungsreaktionen', 'warum': 'Die Carboxylgruppe bleibt reaktionsfähig, der lange unpolare Rest prägt die Stoffeigenschaften.', 'reaktionsidee': 'Salzbildung; Esterbildung.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Diethylenglycol', 'smiles': 'OCCOCCO', 'niveau': 'Vergleich', 'leitfrage': 'Warum ist Diethylenglycol deutlich wasserfreundlicher als Diethylether?', 'beobachten': ['Zähle die Sauerstoffatome.', 'Suche OH-Gruppen.', 'Vergleiche H-Brücken-Donoren und -Akzeptoren.'], 'frage': 'Warum ist Diethylenglycol viel wasserfreundlicher als Diethylether?', 'antworten': ['Es besitzt mehrere O-Atome und zwei OH-Gruppen, kann also H-Brücken spenden und aufnehmen.', 'Es ist wasserfreundlicher, weil es keine Heteroatome besitzt.', 'Diethylether kann mehr H-Brücken spenden als Diethylenglycol.', 'Die zusätzliche OH-Gruppe macht das Molekül vollständig unpolar.'], 'richtig': 0, 'feedback': 'Richtig: Mehrere polare Zentren und OH-Gruppen erhöhen die Wasserwechselwirkung stark.', 'erklaerung': 'Diethylether kann H-Brücken nur aufnehmen. Diethylenglycol kann durch seine OH-Gruppen auch H-Brücken spenden und besitzt eine größere polare Oberfläche.', 'modellgrenze': 'Die Toxizität von Diethylenglycol ist hier nicht Thema; betrachtet wird nur der Struktur–Eigenschaft-Zusammenhang.', 'id': 'diethylenglycol', 'typ': 'molecule', 'modul': 'Polarität und Löslichkeit', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Diethylenglycol deutlich wasserfreundlicher als Diethylether?', 'beobachten': ['Zähle die Sauerstoffatome.', 'Suche OH-Gruppen.', 'Vergleiche H-Brücken-Donoren und -Akzeptoren.'], 'mc_frage': 'Warum ist Diethylenglycol viel wasserfreundlicher als Diethylether?', 'mc_antworten': ['Es besitzt mehrere O-Atome und zwei OH-Gruppen, kann also H-Brücken spenden und aufnehmen.', 'Es ist wasserfreundlicher, weil es keine Heteroatome besitzt.', 'Diethylether kann mehr H-Brücken spenden als Diethylenglycol.', 'Die zusätzliche OH-Gruppe macht das Molekül vollständig unpolar.'], 'mc_richtig': 0, 'feedback': 'Richtig: Mehrere polare Zentren und OH-Gruppen erhöhen die Wasserwechselwirkung stark.', 'erklaerung': 'Diethylether kann H-Brücken nur aufnehmen. Diethylenglycol kann durch seine OH-Gruppen auch H-Brücken spenden und besitzt eine größere polare Oberfläche.', 'modellgrenze': 'Die Toxizität von Diethylenglycol ist hier nicht Thema; betrachtet wird nur der Struktur–Eigenschaft-Zusammenhang.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−10', 'siedepunkt_C': '245', 'dichte_g_cm3': '1.12', 'wasserloeslichkeit': 'sehr gut mischbar/wasserfreundlich', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten; toxikologisch relevant', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Die OH-Gruppen ermöglichen typische Alkoholreaktionen.', 'reaktionsidee': 'Esterbildung; Oxidation unter geeigneten Bedingungen.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}], 'Isomerie': [{'name': 'Ethanol', 'smiles': 'CCO', 'niveau': 'Basis', 'leitfrage': 'Warum hat Ethanol andere Eigenschaften als Dimethylether, obwohl beide C₂H₆O besitzen?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum hat Ethanol andere Stoffeigenschaften als Dimethylether?', 'antworten': ['Ethanol besitzt eine O-H-Gruppe und kann H-Brücken spenden; Dimethylether nicht.', 'Beide müssen gleiche Eigenschaften haben, weil ihre Summenformel gleich ist.', 'Dimethylether ist polarer, weil der Sauerstoff zwischen zwei C-Atomen sitzt.', 'Ethanol und Dimethylether unterscheiden sich nur in der Molmasse.'], 'richtig': 0, 'feedback': 'Richtig: Die Verknüpfung der Atome erzeugt unterschiedliche funktionelle Gruppen.', 'erklaerung': 'Gleiche Summenformel, aber andere Verknüpfung der Atome führt zu anderen funktionellen Gruppen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'ethanol', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum hat Ethanol andere Eigenschaften als Dimethylether, obwohl beide C₂H₆O besitzen?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum hat Ethanol andere Stoffeigenschaften als Dimethylether?', 'mc_antworten': ['Ethanol besitzt eine O-H-Gruppe und kann H-Brücken spenden; Dimethylether nicht.', 'Beide müssen gleiche Eigenschaften haben, weil ihre Summenformel gleich ist.', 'Dimethylether ist polarer, weil der Sauerstoff zwischen zwei C-Atomen sitzt.', 'Ethanol und Dimethylether unterscheiden sich nur in der Molmasse.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die Verknüpfung der Atome erzeugt unterschiedliche funktionelle Gruppen.', 'erklaerung': 'Gleiche Summenformel, aber andere Verknüpfung der Atome führt zu anderen funktionellen Gruppen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−114.1', 'siedepunkt_C': '78.4', 'dichte_g_cm3': '0.789', 'wasserloeslichkeit': 'vollständig mischbar', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Die OH-Gruppe ermöglicht Oxidation, Veresterung und Wasserstoffbrücken.', 'reaktionsidee': 'Oxidation zu Ethanal/Essigsäure; Esterbildung.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Dimethylether', 'smiles': 'COC', 'niveau': 'Basis', 'leitfrage': 'Warum siedet Dimethylether viel niedriger als Ethanol?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum siedet Dimethylether viel niedriger als Ethanol?', 'antworten': ['Dimethylether kann keine H-Brücken zwischen eigenen Molekülen spenden.', 'Dimethylether besitzt keine polaren Bindungen.', 'Dimethylether hat eine viel kleinere molare Masse als Ethanol.', 'Dimethylether ist ein Ionengitter.'], 'richtig': 0, 'feedback': 'Richtig: Der fehlende H-Brücken-Donor ist ein zentraler Unterschied.', 'erklaerung': 'Er besitzt eine Ethergruppe statt einer Alkoholgruppe.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'dimethylether', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum siedet Dimethylether viel niedriger als Ethanol?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum siedet Dimethylether viel niedriger als Ethanol?', 'mc_antworten': ['Dimethylether kann keine H-Brücken zwischen eigenen Molekülen spenden.', 'Dimethylether besitzt keine polaren Bindungen.', 'Dimethylether hat eine viel kleinere molare Masse als Ethanol.', 'Dimethylether ist ein Ionengitter.'], 'mc_richtig': 0, 'feedback': 'Richtig: Der fehlende H-Brücken-Donor ist ein zentraler Unterschied.', 'erklaerung': 'Er besitzt eine Ethergruppe statt einer Alkoholgruppe.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Propanal', 'smiles': 'CCC=O', 'niveau': 'Basis', 'leitfrage': 'Wie unterscheidet sich Propanal von Aceton?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Was unterscheidet Propanal von Aceton?', 'antworten': ['Propanal ist ein Aldehyd; das Carbonyl-C trägt ein H-Atom.', 'Propanal und Aceton besitzen unterschiedliche Summenformeln.', 'Propanal ist ein Alkohol, Aceton ein Ether.', 'Bei Carbonylverbindungen spielt die Position der C=O-Gruppe keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Aldehyd und Keton unterscheiden sich in der Umgebung der Carbonylgruppe.', 'erklaerung': 'Die Carbonylgruppe liegt am Ende der Kette.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'propanal', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Wie unterscheidet sich Propanal von Aceton?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Was unterscheidet Propanal von Aceton?', 'mc_antworten': ['Propanal ist ein Aldehyd; das Carbonyl-C trägt ein H-Atom.', 'Propanal und Aceton besitzen unterschiedliche Summenformeln.', 'Propanal ist ein Alkohol, Aceton ein Ether.', 'Bei Carbonylverbindungen spielt die Position der C=O-Gruppe keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Aldehyd und Keton unterscheiden sich in der Umgebung der Carbonylgruppe.', 'erklaerung': 'Die Carbonylgruppe liegt am Ende der Kette.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Aceton', 'smiles': 'CC(=O)C', 'niveau': 'Basis', 'leitfrage': 'Wie unterscheidet sich Aceton von Propanal?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage beschreibt Aceton als Lösungsmittel am besten?', 'antworten': ['Aceton ist polar aprotisch: es kann H-Brücken aufnehmen, aber keine spenden.', 'Aceton ist ein Alkohol und kann deshalb H-Brücken spenden.', 'Aceton ist unpolar, weil es drei Kohlenstoffatome enthält.', 'Aceton besitzt eine Carboxylgruppe und reagiert sauer.'], 'richtig': 0, 'feedback': 'Richtig: Die C=O-Gruppe macht Aceton polar, aber ohne O-H/N-H kann es keine H-Brücken spenden.', 'erklaerung': 'Die Carbonylgruppe liegt zwischen zwei Kohlenstoffatomen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'aceton', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Wie unterscheidet sich Aceton von Propanal?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage beschreibt Aceton als Lösungsmittel am besten?', 'mc_antworten': ['Aceton ist polar aprotisch: es kann H-Brücken aufnehmen, aber keine spenden.', 'Aceton ist ein Alkohol und kann deshalb H-Brücken spenden.', 'Aceton ist unpolar, weil es drei Kohlenstoffatome enthält.', 'Aceton besitzt eine Carboxylgruppe und reagiert sauer.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die C=O-Gruppe macht Aceton polar, aber ohne O-H/N-H kann es keine H-Brücken spenden.', 'erklaerung': 'Die Carbonylgruppe liegt zwischen zwei Kohlenstoffatomen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−95', 'siedepunkt_C': '56.1', 'dichte_g_cm3': '0.79', 'wasserloeslichkeit': 'vollständig mischbar', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Die Carbonylgruppe ist elektrophil, aber Ketone sind gegenüber Oxidation stabiler als Aldehyde.', 'reaktionsidee': 'Nukleophile Addition an die C=O-Gruppe.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Butan', 'smiles': 'CCCC', 'niveau': 'Basis', 'leitfrage': 'Was unterscheidet Butan von Isobutan?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Was unterscheidet Butan von Isobutan?', 'antworten': ['Die C-Atome sind anders verknüpft: Butan ist unverzweigt, Isobutan verzweigt.', 'Butan und Isobutan unterscheiden sich in der Summenformel.', 'Isobutan enthält eine OH-Gruppe.', 'Verzweigung verändert keine physikalischen Eigenschaften.'], 'richtig': 0, 'feedback': 'Richtig: Es handelt sich um Konstitutionsisomerie mit anderer C-Gerüstform.', 'erklaerung': 'Beide haben dieselbe Summenformel, aber unterschiedliche Kohlenstoffverknüpfung.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'butan', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Was unterscheidet Butan von Isobutan?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Was unterscheidet Butan von Isobutan?', 'mc_antworten': ['Die C-Atome sind anders verknüpft: Butan ist unverzweigt, Isobutan verzweigt.', 'Butan und Isobutan unterscheiden sich in der Summenformel.', 'Isobutan enthält eine OH-Gruppe.', 'Verzweigung verändert keine physikalischen Eigenschaften.'], 'mc_richtig': 0, 'feedback': 'Richtig: Es handelt sich um Konstitutionsisomerie mit anderer C-Gerüstform.', 'erklaerung': 'Beide haben dieselbe Summenformel, aber unterschiedliche Kohlenstoffverknüpfung.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Isobutan', 'smiles': 'CC(C)C', 'niveau': 'Basis', 'leitfrage': 'Warum ist Isobutan kompakter als Butan?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum kann Verzweigung Stoffeigenschaften verändern?', 'antworten': ['Die Molekülform und Kontaktfläche zwischen Molekülen ändern sich.', 'Verzweigung erzeugt automatisch Ionen.', 'Verzweigung ändert immer die Summenformel.', 'Verzweigte Moleküle können keine London-Kräfte bilden.'], 'richtig': 0, 'feedback': 'Richtig: Verzweigung beeinflusst Form, Oberfläche und zwischenmolekulare Wechselwirkungen.', 'erklaerung': 'Verzweigung verändert Molekülform und damit Stoffeigenschaften.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'isobutan', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Isobutan kompakter als Butan?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum kann Verzweigung Stoffeigenschaften verändern?', 'mc_antworten': ['Die Molekülform und Kontaktfläche zwischen Molekülen ändern sich.', 'Verzweigung erzeugt automatisch Ionen.', 'Verzweigung ändert immer die Summenformel.', 'Verzweigte Moleküle können keine London-Kräfte bilden.'], 'mc_richtig': 0, 'feedback': 'Richtig: Verzweigung beeinflusst Form, Oberfläche und zwischenmolekulare Wechselwirkungen.', 'erklaerung': 'Verzweigung verändert Molekülform und damit Stoffeigenschaften.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Propansäure', 'smiles': 'CCC(=O)O', 'niveau': 'Basis', 'leitfrage': 'Warum unterscheidet sich Propansäure stark von Essigsäuremethylester?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Propansäure nicht mit Essigsäuremethylester gleichzusetzen?', 'antworten': ['Propansäure besitzt eine Carboxylgruppe mit saurem O-H; der Ester nicht.', 'Beide haben dieselbe funktionelle Gruppe.', 'Der Ester ist eine Carbonsäure, weil er zwei Sauerstoffatome enthält.', 'Nur die Summenformel entscheidet über Säureeigenschaften.'], 'richtig': 0, 'feedback': 'Richtig: Carbonsäure und Ester haben unterschiedliche funktionelle Gruppen.', 'erklaerung': 'Funktionelle Isomerie: Carbonsäure vs. Ester.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'propansaeure', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum unterscheidet sich Propansäure stark von Essigsäuremethylester?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Propansäure nicht mit Essigsäuremethylester gleichzusetzen?', 'mc_antworten': ['Propansäure besitzt eine Carboxylgruppe mit saurem O-H; der Ester nicht.', 'Beide haben dieselbe funktionelle Gruppe.', 'Der Ester ist eine Carbonsäure, weil er zwei Sauerstoffatome enthält.', 'Nur die Summenformel entscheidet über Säureeigenschaften.'], 'mc_richtig': 0, 'feedback': 'Richtig: Carbonsäure und Ester haben unterschiedliche funktionelle Gruppen.', 'erklaerung': 'Funktionelle Isomerie: Carbonsäure vs. Ester.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Essigsäuremethylester', 'smiles': 'CC(=O)OC', 'niveau': 'Basis', 'leitfrage': 'Warum ist Essigsäuremethylester kein Säure-Isomer im Eigenschaftssinn?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum zeigt ein Ester andere Eigenschaften als eine Carbonsäure?', 'antworten': ['Beim Ester fehlt die saure O-H-Gruppe der Carboxylgruppe.', 'Ester enthalten keine Sauerstoffatome.', 'Ester sind immer ionisch.', 'Carbonsäure und Ester müssen gleich reagieren, wenn die Summenformel passt.'], 'richtig': 0, 'feedback': 'Richtig: Der Austausch der O-H-Gruppe verändert Polarität und Reaktivität stark.', 'erklaerung': 'Gleiche Summenformel kann durch andere funktionelle Gruppe ganz andere Eigenschaften ergeben.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'essigsaeuremethylester', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Essigsäuremethylester kein Säure-Isomer im Eigenschaftssinn?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum zeigt ein Ester andere Eigenschaften als eine Carbonsäure?', 'mc_antworten': ['Beim Ester fehlt die saure O-H-Gruppe der Carboxylgruppe.', 'Ester enthalten keine Sauerstoffatome.', 'Ester sind immer ionisch.', 'Carbonsäure und Ester müssen gleich reagieren, wenn die Summenformel passt.'], 'mc_richtig': 0, 'feedback': 'Richtig: Der Austausch der O-H-Gruppe verändert Polarität und Reaktivität stark.', 'erklaerung': 'Gleiche Summenformel kann durch andere funktionelle Gruppe ganz andere Eigenschaften ergeben.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'D-Alanin', 'smiles': 'C[C@@H](N)C(=O)O', 'niveau': 'Basis', 'leitfrage': 'Warum sind D- und L-Alanin trotz gleicher Verknüpfung nicht dasselbe Molekül?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Was zeigt der Vergleich von D- und L-Alanin?', 'antworten': ['Gleiche Verknüpfung kann durch räumliche Spiegelbildlichkeit zu unterschiedlichen biologischen Wirkungen führen.', 'D- und L-Alanin unterscheiden sich in der Summenformel.', 'D-Alanin enthält eine andere funktionelle Gruppe als L-Alanin.', 'Spiegelbildlichkeit spielt bei Biomolekülen keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Enantiomere sind spiegelbildlich und in chiralen Umgebungen oft verschieden wirksam.', 'erklaerung': 'Chiralität zeigt, dass auch die räumliche Anordnung entscheidend ist.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'd_alanin', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum sind D- und L-Alanin trotz gleicher Verknüpfung nicht dasselbe Molekül?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Was zeigt der Vergleich von D- und L-Alanin?', 'mc_antworten': ['Gleiche Verknüpfung kann durch räumliche Spiegelbildlichkeit zu unterschiedlichen biologischen Wirkungen führen.', 'D- und L-Alanin unterscheiden sich in der Summenformel.', 'D-Alanin enthält eine andere funktionelle Gruppe als L-Alanin.', 'Spiegelbildlichkeit spielt bei Biomolekülen keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Enantiomere sind spiegelbildlich und in chiralen Umgebungen oft verschieden wirksam.', 'erklaerung': 'Chiralität zeigt, dass auch die räumliche Anordnung entscheidend ist.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'L-Alanin', 'smiles': 'C[C@H](N)C(=O)O', 'niveau': 'Basis', 'leitfrage': 'Warum ist Chiralität bei Aminosäuren biologisch wichtig?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['L-Alanin ist eine spiegelbildliche Form von D-Alanin.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: L-Alanin ist eine spiegelbildliche Form von D-Alanin.', 'erklaerung': 'Biologische Systeme unterscheiden Spiegelbilder häufig sehr genau.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'l_alanin', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Chiralität bei Aminosäuren biologisch wichtig?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['L-Alanin ist eine spiegelbildliche Form von D-Alanin.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: L-Alanin ist eine spiegelbildliche Form von D-Alanin.', 'erklaerung': 'Biologische Systeme unterscheiden Spiegelbilder häufig sehr genau.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Glucose offen', 'smiles': 'O=CC(O)C(O)C(O)C(O)CO', 'niveau': 'Basis', 'leitfrage': 'Warum ist Glucose ein komplexeres Isomeriebeispiel?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum sind Glucose und Fructose trotz gleicher Summenformel verschieden?', 'antworten': ['Glucose ist in der offenkettigen Form eine Aldose, Fructose eine Ketose.', 'Beide besitzen dieselbe funktionelle Gruppe an derselben Stelle.', 'Die Lage der C=O-Gruppe beeinflusst Eigenschaften nicht.', 'Zucker unterscheiden sich nur durch die Anzahl der C-Atome.'], 'richtig': 0, 'feedback': 'Richtig: Die Position und Art der Carbonylgruppe ist entscheidend.', 'erklaerung': 'Glucose und Fructose haben dieselbe Summenformel, unterscheiden sich aber in der Carbonylgruppe.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'glucose_offen', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Glucose ein komplexeres Isomeriebeispiel?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum sind Glucose und Fructose trotz gleicher Summenformel verschieden?', 'mc_antworten': ['Glucose ist in der offenkettigen Form eine Aldose, Fructose eine Ketose.', 'Beide besitzen dieselbe funktionelle Gruppe an derselben Stelle.', 'Die Lage der C=O-Gruppe beeinflusst Eigenschaften nicht.', 'Zucker unterscheiden sich nur durch die Anzahl der C-Atome.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die Position und Art der Carbonylgruppe ist entscheidend.', 'erklaerung': 'Glucose und Fructose haben dieselbe Summenformel, unterscheiden sich aber in der Carbonylgruppe.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Fructose offen', 'smiles': 'OCC(O)C(O)C(O)C(=O)CO', 'niveau': 'Basis', 'leitfrage': 'Warum ist Fructose trotz gleicher Summenformel anders als Glucose?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Fructose ist eine Ketose.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Fructose ist eine Ketose.', 'erklaerung': 'In Lösung sind zusätzlich Ringformen wichtig; hier wird bewusst vereinfacht.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'fructose_offen', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Fructose trotz gleicher Summenformel anders als Glucose?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Fructose ist eine Ketose.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Fructose ist eine Ketose.', 'erklaerung': 'In Lösung sind zusätzlich Ringformen wichtig; hier wird bewusst vereinfacht.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Maleinsäure', 'smiles': 'O=C(O)/C=C\\C(=O)O', 'niveau': 'Vertiefung', 'leitfrage': 'Warum kann die cis/trans-Anordnung die Stoffeigenschaften stark verändern?', 'beobachten': ['Vergleiche die Lage der beiden Carboxylgruppen.', 'Achte auf die räumliche Nähe der Gruppen.', 'Vergleiche mit Fumarsäure im Isomerenvergleich.'], 'frage': 'Warum kann cis/trans-Isomerie die Eigenschaften stark verändern?', 'antworten': ['Die räumliche Lage der Carboxylgruppen verändert Wechselwirkungen und Stoffeigenschaften.', 'cis/trans-Isomere haben immer verschiedene Summenformeln.', 'Doppelbindungen sind frei drehbar wie Einfachbindungen.', 'Maleinsäure und Fumarsäure sind nur verschiedene Namen für dasselbe Molekül.'], 'richtig': 0, 'feedback': 'Richtig: Die starre C=C-Doppelbindung fixiert unterschiedliche räumliche Anordnungen.', 'erklaerung': 'Maleinsäure ist das cis-Isomer; die Carboxylgruppen liegen räumlich nahe beieinander. Dadurch unterscheiden sich Eigenschaften wie Löslichkeit, Schmelzpunkt und mögliche intramolekulare Wechselwirkungen von Fumarsäure.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'maleinsaeure', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum kann die cis/trans-Anordnung die Stoffeigenschaften stark verändern?', 'beobachten': ['Vergleiche die Lage der beiden Carboxylgruppen.', 'Achte auf die räumliche Nähe der Gruppen.', 'Vergleiche mit Fumarsäure im Isomerenvergleich.'], 'mc_frage': 'Warum kann cis/trans-Isomerie die Eigenschaften stark verändern?', 'mc_antworten': ['Die räumliche Lage der Carboxylgruppen verändert Wechselwirkungen und Stoffeigenschaften.', 'cis/trans-Isomere haben immer verschiedene Summenformeln.', 'Doppelbindungen sind frei drehbar wie Einfachbindungen.', 'Maleinsäure und Fumarsäure sind nur verschiedene Namen für dasselbe Molekül.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die starre C=C-Doppelbindung fixiert unterschiedliche räumliche Anordnungen.', 'erklaerung': 'Maleinsäure ist das cis-Isomer; die Carboxylgruppen liegen räumlich nahe beieinander. Dadurch unterscheiden sich Eigenschaften wie Löslichkeit, Schmelzpunkt und mögliche intramolekulare Wechselwirkungen von Fumarsäure.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': 'ca. 130–135', 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': 'gut löslich', 'aggregatzustand_raumtemperatur': 'fest', 'bedingungen': 'ca. 20–25 °C', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv', 'warum': 'Zwei Carboxylgruppen und die C=C-Doppelbindung bieten mehrere Reaktionsmöglichkeiten.', 'reaktionsidee': 'Säure-Base-Reaktionen; Additionen an C=C unter geeigneten Bedingungen.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Fumarsäure', 'smiles': 'O=C(O)/C=C/C(=O)O', 'niveau': 'Vertiefung', 'leitfrage': 'Warum ist Fumarsäure trotz gleicher Summenformel anders als Maleinsäure?', 'beobachten': ['Vergleiche die Lage der Carboxylgruppen.', 'Achte auf die trans-Anordnung.', 'Überlege, warum das die Wechselwirkungen verändert.'], 'frage': 'Warum ist Fumarsäure nicht einfach identisch mit Maleinsäure?', 'antworten': ['Fumarsäure ist das trans-Isomer; die Carboxylgruppen liegen auf gegenüberliegenden Seiten.', 'Fumarsäure hat keine Doppelbindung.', 'Fumarsäure enthält keine Carboxylgruppen.', 'trans-Isomere sind immer unpolar.'], 'richtig': 0, 'feedback': 'Richtig: Die trans-Anordnung verändert Wechselwirkungen und Stoffeigenschaften.', 'erklaerung': 'Fumarsäure besitzt dieselbe Summenformel wie Maleinsäure, aber eine andere räumliche Anordnung. Diese scheinbar kleine Änderung kann deutliche Stoffeigenschaftsunterschiede bewirken.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'fumarsaeure', 'typ': 'molecule', 'modul': 'Isomerie', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Fumarsäure trotz gleicher Summenformel anders als Maleinsäure?', 'beobachten': ['Vergleiche die Lage der Carboxylgruppen.', 'Achte auf die trans-Anordnung.', 'Überlege, warum das die Wechselwirkungen verändert.'], 'mc_frage': 'Warum ist Fumarsäure nicht einfach identisch mit Maleinsäure?', 'mc_antworten': ['Fumarsäure ist das trans-Isomer; die Carboxylgruppen liegen auf gegenüberliegenden Seiten.', 'Fumarsäure hat keine Doppelbindung.', 'Fumarsäure enthält keine Carboxylgruppen.', 'trans-Isomere sind immer unpolar.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die trans-Anordnung verändert Wechselwirkungen und Stoffeigenschaften.', 'erklaerung': 'Fumarsäure besitzt dieselbe Summenformel wie Maleinsäure, aber eine andere räumliche Anordnung. Diese scheinbar kleine Änderung kann deutliche Stoffeigenschaftsunterschiede bewirken.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '>280 (Zersetzung/Sublimation)', 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': 'schlecht löslich', 'aggregatzustand_raumtemperatur': 'fest', 'bedingungen': 'ca. 20–25 °C', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv', 'warum': 'Zwei Carboxylgruppen und die C=C-Doppelbindung bieten mehrere Reaktionsmöglichkeiten.', 'reaktionsidee': 'Säure-Base-Reaktionen; Additionen an C=C unter geeigneten Bedingungen.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}], 'Funktionelle Gruppen': [{'name': 'Methanol', 'smiles': 'CO', 'niveau': 'Basis', 'leitfrage': 'Woran erkennt man einen Alkohol?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Methanol sehr gut mit Wasser mischbar?', 'antworten': ['Die OH-Gruppe bildet H-Brücken und der unpolare Rest ist sehr klein.', 'Methanol ist gut löslich, weil alle organischen Moleküle mit Sauerstoff vollständig wasserlöslich sind.', 'Methanol enthält eine Carbonylgruppe, die stark sauer reagiert.', 'Die C-H-Bindungen sind der Hauptgrund für die Wasserlöslichkeit.'], 'richtig': 0, 'feedback': 'Richtig: Die OH-Gruppe dominiert hier die Eigenschaften, weil der unpolare Teil klein ist.', 'erklaerung': 'Eine OH-Gruppe an einem gesättigten Kohlenstoff prägt Polarität und H-Brücken.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'methanol', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Woran erkennt man einen Alkohol?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Methanol sehr gut mit Wasser mischbar?', 'mc_antworten': ['Die OH-Gruppe bildet H-Brücken und der unpolare Rest ist sehr klein.', 'Methanol ist gut löslich, weil alle organischen Moleküle mit Sauerstoff vollständig wasserlöslich sind.', 'Methanol enthält eine Carbonylgruppe, die stark sauer reagiert.', 'Die C-H-Bindungen sind der Hauptgrund für die Wasserlöslichkeit.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die OH-Gruppe dominiert hier die Eigenschaften, weil der unpolare Teil klein ist.', 'erklaerung': 'Eine OH-Gruppe an einem gesättigten Kohlenstoff prägt Polarität und H-Brücken.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−97.6', 'siedepunkt_C': '64.7', 'dichte_g_cm3': '0.79', 'wasserloeslichkeit': 'vollständig mischbar', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Die OH-Gruppe ermöglicht Oxidation, Veresterung und Säure-Base-Wechselwirkungen.', 'reaktionsidee': 'Oxidation zu Methanal; Bildung von Estern.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Ethanal', 'smiles': 'CC=O', 'niveau': 'Basis', 'leitfrage': 'Woran erkennt man einen Aldehyd?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Ethanal besitzt eine Aldehydgruppe.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Ethanal besitzt eine Aldehydgruppe.', 'erklaerung': 'Die Carbonylgruppe liegt am Kettenende.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'ethanal', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Woran erkennt man einen Aldehyd?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Ethanal besitzt eine Aldehydgruppe.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Ethanal besitzt eine Aldehydgruppe.', 'erklaerung': 'Die Carbonylgruppe liegt am Kettenende.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv', 'warum': 'Aldehyde besitzen eine elektrophile Carbonylgruppe und sind leicht oxidierbar.', 'reaktionsidee': 'Nukleophile Addition; Oxidation zu Carbonsäuren.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Aceton', 'smiles': 'CC(=O)C', 'niveau': 'Basis', 'leitfrage': 'Woran erkennt man ein Keton?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage beschreibt Aceton als Lösungsmittel am besten?', 'antworten': ['Aceton ist polar aprotisch: es kann H-Brücken aufnehmen, aber keine spenden.', 'Aceton ist ein Alkohol und kann deshalb H-Brücken spenden.', 'Aceton ist unpolar, weil es drei Kohlenstoffatome enthält.', 'Aceton besitzt eine Carboxylgruppe und reagiert sauer.'], 'richtig': 0, 'feedback': 'Richtig: Die C=O-Gruppe macht Aceton polar, aber ohne O-H/N-H kann es keine H-Brücken spenden.', 'erklaerung': 'Die Carbonylgruppe liegt zwischen zwei Kohlenstoffresten.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'aceton', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Woran erkennt man ein Keton?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage beschreibt Aceton als Lösungsmittel am besten?', 'mc_antworten': ['Aceton ist polar aprotisch: es kann H-Brücken aufnehmen, aber keine spenden.', 'Aceton ist ein Alkohol und kann deshalb H-Brücken spenden.', 'Aceton ist unpolar, weil es drei Kohlenstoffatome enthält.', 'Aceton besitzt eine Carboxylgruppe und reagiert sauer.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die C=O-Gruppe macht Aceton polar, aber ohne O-H/N-H kann es keine H-Brücken spenden.', 'erklaerung': 'Die Carbonylgruppe liegt zwischen zwei Kohlenstoffresten.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−95', 'siedepunkt_C': '56.1', 'dichte_g_cm3': '0.79', 'wasserloeslichkeit': 'vollständig mischbar', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Die Carbonylgruppe ist elektrophil, aber Ketone sind gegenüber Oxidation stabiler als Aldehyde.', 'reaktionsidee': 'Nukleophile Addition an die C=O-Gruppe.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Essigsäure', 'smiles': 'CC(=O)O', 'niveau': 'Basis', 'leitfrage': 'Woran erkennt man eine Carbonsäure?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Essigsäure besonders gut für Struktur-Eigenschafts-Vergleiche geeignet?', 'antworten': ['Die Carboxylgruppe verbindet Polarität, H-Brücken und Säure-Base-Verhalten.', 'Essigsäure ist ein völlig unpolares Molekül.', 'Essigsäure enthält nur eine Ethergruppe.', 'Die Methylgruppe allein erklärt die hohe Wasserlöslichkeit.'], 'richtig': 0, 'feedback': 'Richtig: Die Carboxylgruppe ist der zentrale Strukturbaustein für die Eigenschaften.', 'erklaerung': 'C=O und OH am selben Kohlenstoffatom bilden die Carboxylgruppe.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'essigsaeure', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Woran erkennt man eine Carbonsäure?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Essigsäure besonders gut für Struktur-Eigenschafts-Vergleiche geeignet?', 'mc_antworten': ['Die Carboxylgruppe verbindet Polarität, H-Brücken und Säure-Base-Verhalten.', 'Essigsäure ist ein völlig unpolares Molekül.', 'Essigsäure enthält nur eine Ethergruppe.', 'Die Methylgruppe allein erklärt die hohe Wasserlöslichkeit.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die Carboxylgruppe ist der zentrale Strukturbaustein für die Eigenschaften.', 'erklaerung': 'C=O und OH am selben Kohlenstoffatom bilden die Carboxylgruppe.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '16.6', 'siedepunkt_C': '118.1', 'dichte_g_cm3': '1.05', 'wasserloeslichkeit': 'vollständig mischbar', 'aggregatzustand_raumtemperatur': 'flüssig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv in Säure-Base-Reaktionen', 'warum': 'Die Carboxylgruppe kann ein Proton abgeben.', 'reaktionsidee': 'Neutralisation mit Basen; Esterbildung mit Alkoholen.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Ethylacetat', 'smiles': 'CC(=O)OCC', 'niveau': 'Basis', 'leitfrage': 'Woran erkennt man einen Ester?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Ethylacetat besitzt eine Estergruppe.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Ethylacetat besitzt eine Estergruppe.', 'erklaerung': 'Das Strukturmotiv ist C(=O)-O-C.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'ethylacetat', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Woran erkennt man einen Ester?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Ethylacetat besitzt eine Estergruppe.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Ethylacetat besitzt eine Estergruppe.', 'erklaerung': 'Das Strukturmotiv ist C(=O)-O-C.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Ester können hydrolysiert werden, sind aber weniger reaktiv als Säureanhydride.', 'reaktionsidee': 'Saure oder basische Esterhydrolyse.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Diethylether', 'smiles': 'CCOCC', 'niveau': 'Basis', 'leitfrage': 'Woran erkennt man einen Ether?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Warum ist Diethylether deutlich anders als Ethanol?', 'antworten': ['Er kann H-Brücken aufnehmen, aber mangels O-H-Gruppe nicht spenden.', 'Er enthält keine polaren Bindungen.', 'Er ist wegen des Sauerstoffatoms automatisch vollständig mit Wasser mischbar.', 'Er besitzt eine Carboxylgruppe und reagiert sauer.'], 'richtig': 0, 'feedback': 'Richtig: Ether-Sauerstoff kann H-Brücken aufnehmen, aber Diethylether kann keine spenden.', 'erklaerung': 'Ein Sauerstoffatom ist mit zwei Kohlenstoffresten verbunden.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'diethylether', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Woran erkennt man einen Ether?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Warum ist Diethylether deutlich anders als Ethanol?', 'mc_antworten': ['Er kann H-Brücken aufnehmen, aber mangels O-H-Gruppe nicht spenden.', 'Er enthält keine polaren Bindungen.', 'Er ist wegen des Sauerstoffatoms automatisch vollständig mit Wasser mischbar.', 'Er besitzt eine Carboxylgruppe und reagiert sauer.'], 'mc_richtig': 0, 'feedback': 'Richtig: Ether-Sauerstoff kann H-Brücken aufnehmen, aber Diethylether kann keine spenden.', 'erklaerung': 'Ein Sauerstoffatom ist mit zwei Kohlenstoffresten verbunden.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': '−116', 'siedepunkt_C': '34.6', 'dichte_g_cm3': '0.71', 'wasserloeslichkeit': 'begrenzt löslich', 'aggregatzustand_raumtemperatur': 'flüssig, sehr flüchtig', 'bedingungen': 'ca. 20–25 °C, 1 atm', 'quelle': 'kuratierte Unterrichtsdaten', 'hinweis': None}, 'reaktivitaet': {'hinweis': 'relativ reaktionsträge', 'warum': 'Ether besitzen keine O–H-Bindung und reagieren unter milden Bedingungen meist weniger stark als Alkohole.', 'reaktionsidee': 'Spaltung unter starken sauren Bedingungen; Peroxidbildung als Sicherheitsaspekt.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Acetanhydrid', 'smiles': 'CC(=O)OC(=O)C', 'niveau': 'Basis', 'leitfrage': 'Woran erkennt man ein Carbonsäureanhydrid?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Acetanhydrid besitzt eine Anhydridgruppe.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Acetanhydrid besitzt eine Anhydridgruppe.', 'erklaerung': 'Zwei Acylgruppen sind über ein Sauerstoffatom verbunden.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'acetanhydrid', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Woran erkennt man ein Carbonsäureanhydrid?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Acetanhydrid besitzt eine Anhydridgruppe.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Acetanhydrid besitzt eine Anhydridgruppe.', 'erklaerung': 'Zwei Acylgruppen sind über ein Sauerstoffatom verbunden.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv', 'warum': 'Säureanhydride reagieren leicht mit Wasser, Alkoholen und Aminen.', 'reaktionsidee': 'Acylierung; Hydrolyse zu Carbonsäuren.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Ethylamin', 'smiles': 'CCN', 'niveau': 'Basis', 'leitfrage': 'Woran erkennt man ein Amin?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Ethylamin besitzt eine Aminogruppe.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Ethylamin besitzt eine Aminogruppe.', 'erklaerung': 'Amine enthalten Stickstoff und können basische Eigenschaften zeigen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'ethylamin', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Woran erkennt man ein Amin?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Ethylamin besitzt eine Aminogruppe.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Ethylamin besitzt eine Aminogruppe.', 'erklaerung': 'Amine enthalten Stickstoff und können basische Eigenschaften zeigen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv als Base/Nukleophil', 'warum': 'Das freie Elektronenpaar am Stickstoff kann Protonen aufnehmen und Bindungen bilden.', 'reaktionsidee': 'Bildung von Ammoniumsalzen; nukleophile Reaktionen.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Acetamid', 'smiles': 'CC(=O)N', 'niveau': 'Basis', 'leitfrage': 'Woran erkennt man ein Amid?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Acetamid besitzt eine Amidgruppe.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Acetamid besitzt eine Amidgruppe.', 'erklaerung': 'C(=O)-N ist das zentrale Motiv; die Peptidbindung ist ebenfalls ein Amid.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'acetamid', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Woran erkennt man ein Amid?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Acetamid besitzt eine Amidgruppe.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Acetamid besitzt eine Amidgruppe.', 'erklaerung': 'C(=O)-N ist das zentrale Motiv; die Peptidbindung ist ebenfalls ein Amid.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'gering bis mäßig reaktiv', 'warum': 'Die Amidbindung ist durch Mesomerie stabilisiert.', 'reaktionsidee': 'Hydrolyse unter kräftigen sauren oder basischen Bedingungen.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Phenol', 'smiles': 'c1ccccc1O', 'niveau': 'Basis', 'leitfrage': 'Warum ist Phenol nicht einfach ein normaler Alkohol?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Die OH-Gruppe ist direkt an einen aromatischen Ring gebunden.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Die OH-Gruppe ist direkt an einen aromatischen Ring gebunden.', 'erklaerung': 'Phenole unterscheiden sich in Eigenschaften und Reaktivität von einfachen Alkoholen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'phenol', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Phenol nicht einfach ein normaler Alkohol?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Die OH-Gruppe ist direkt an einen aromatischen Ring gebunden.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die OH-Gruppe ist direkt an einen aromatischen Ring gebunden.', 'erklaerung': 'Phenole unterscheiden sich in Eigenschaften und Reaktivität von einfachen Alkoholen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Phenol ist schwach sauer und der aromatische Ring ist beeinflusst durch die OH-Gruppe.', 'reaktionsidee': 'Bildung von Phenolaten; elektrophile aromatische Substitution.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Anilin', 'smiles': 'c1ccccc1N', 'niveau': 'Basis', 'leitfrage': 'Warum ist Anilin ein aromatisches Amin?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Anilin besitzt eine Aminogruppe am aromatischen Ring.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Anilin besitzt eine Aminogruppe am aromatischen Ring.', 'erklaerung': 'Aromat und Aminogruppe beeinflussen sich gegenseitig.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'anilin', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Anilin ein aromatisches Amin?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Anilin besitzt eine Aminogruppe am aromatischen Ring.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Anilin besitzt eine Aminogruppe am aromatischen Ring.', 'erklaerung': 'Aromat und Aminogruppe beeinflussen sich gegenseitig.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Die Aminogruppe ist basisch/nukleophil und aktiviert den aromatischen Ring.', 'reaktionsidee': 'Salzbildung; elektrophile aromatische Substitution.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Salicylsäure', 'smiles': 'O=C(O)c1ccccc1O', 'niveau': 'Vertiefung', 'leitfrage': 'Welche funktionellen Gruppen erkennt man in Salicylsäure?', 'beobachten': ['Suche die Carboxylgruppe.', 'Suche die phenolische OH-Gruppe.', 'Achte auf den aromatischen Ring.'], 'frage': 'Warum ist Salicylsäure für funktionelle Gruppen besonders interessant?', 'antworten': ['Carboxylgruppe, Phenol-OH und Aromat liegen in einem kleinen Molekül zusammen.', 'Salicylsäure enthält nur Alkaneigenschaften.', 'Salicylsäure ist ein Ether ohne OH-Gruppe.', 'Der aromatische Ring verhindert jede Polarität.'], 'richtig': 0, 'feedback': 'Richtig: Mehrere funktionelle Gruppen beeinflussen Polarität und Reaktivität gleichzeitig.', 'erklaerung': 'Salicylsäure ist ein gutes Beispiel dafür, dass ein Molekül mehrere funktionelle Gruppen tragen kann. Diese Gruppen bestimmen gemeinsam Polarität, Säureeigenschaften und Wechselwirkungen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'salicylsaeure', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Welche funktionellen Gruppen erkennt man in Salicylsäure?', 'beobachten': ['Suche die Carboxylgruppe.', 'Suche die phenolische OH-Gruppe.', 'Achte auf den aromatischen Ring.'], 'mc_frage': 'Warum ist Salicylsäure für funktionelle Gruppen besonders interessant?', 'mc_antworten': ['Carboxylgruppe, Phenol-OH und Aromat liegen in einem kleinen Molekül zusammen.', 'Salicylsäure enthält nur Alkaneigenschaften.', 'Salicylsäure ist ein Ether ohne OH-Gruppe.', 'Der aromatische Ring verhindert jede Polarität.'], 'mc_richtig': 0, 'feedback': 'Richtig: Mehrere funktionelle Gruppen beeinflussen Polarität und Reaktivität gleichzeitig.', 'erklaerung': 'Salicylsäure ist ein gutes Beispiel dafür, dass ein Molekül mehrere funktionelle Gruppen tragen kann. Diese Gruppen bestimmen gemeinsam Polarität, Säureeigenschaften und Wechselwirkungen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'reaktiv in Säure-Base- und Esterbildungsreaktionen', 'warum': 'Carboxylgruppe und Phenolgruppe ermöglichen mehrere Reaktionswege.', 'reaktionsidee': 'Salzbildung; Esterbildung; intramolekulare Wechselwirkungen.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}, {'name': 'Vanillin', 'smiles': 'COc1cc(C=O)ccc1O', 'niveau': 'Vertiefung', 'leitfrage': 'Warum ist Vanillin ein gutes Beispiel für mehrere funktionelle Gruppen in einem Alltagsmolekül?', 'beobachten': ['Suche die Aldehydgruppe.', 'Suche die Methoxy-/Ethergruppe.', 'Suche die phenolische OH-Gruppe und den aromatischen Ring.'], 'frage': 'Welche Aussage erklärt Vanillin als gutes Beispiel für funktionelle Gruppen?', 'antworten': ['Ein bekanntes Molekül kann mehrere funktionelle Gruppen gleichzeitig besitzen.', 'Vanillin ist ein einfacher Kohlenwasserstoff ohne Heteroatome.', 'Vanillin enthält nur eine einzige C-C-Einfachbindung.', 'Die funktionellen Gruppen spielen für Geruch und Reaktivität keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Aldehyd, Phenol, Methoxygruppe und Aromat prägen gemeinsam die Eigenschaften.', 'erklaerung': 'Vanillin zeigt, wie mehrere funktionelle Gruppen zusammen ein charakteristisches Molekül ergeben. Die Struktur erklärt, warum es polarere Bereiche besitzt, aber auch einen aromatischen, eher unpolaren Bereich.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'vanillin', 'typ': 'molecule', 'modul': 'Funktionelle Gruppen', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Vanillin ein gutes Beispiel für mehrere funktionelle Gruppen in einem Alltagsmolekül?', 'beobachten': ['Suche die Aldehydgruppe.', 'Suche die Methoxy-/Ethergruppe.', 'Suche die phenolische OH-Gruppe und den aromatischen Ring.'], 'mc_frage': 'Welche Aussage erklärt Vanillin als gutes Beispiel für funktionelle Gruppen?', 'mc_antworten': ['Ein bekanntes Molekül kann mehrere funktionelle Gruppen gleichzeitig besitzen.', 'Vanillin ist ein einfacher Kohlenwasserstoff ohne Heteroatome.', 'Vanillin enthält nur eine einzige C-C-Einfachbindung.', 'Die funktionellen Gruppen spielen für Geruch und Reaktivität keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Aldehyd, Phenol, Methoxygruppe und Aromat prägen gemeinsam die Eigenschaften.', 'erklaerung': 'Vanillin zeigt, wie mehrere funktionelle Gruppen zusammen ein charakteristisches Molekül ergeben. Die Struktur erklärt, warum es polarere Bereiche besitzt, aber auch einen aromatischen, eher unpolaren Bereich.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': 'mäßig reaktiv', 'warum': 'Aldehyd, Phenol und Methoxygruppe bieten verschiedene Reaktionsmöglichkeiten.', 'reaktionsidee': 'Reaktionen der Aldehydgruppe; Reaktionen am aromatischen Ring.', 'modellgrenze': 'Dieser Hinweis ist eine qualitative Orientierung anhand funktioneller Gruppen und ersetzt keine vollständige Reaktionslehre.'}}], 'Konformere': [{'name': 'Butan', 'smiles': 'CCCC', 'niveau': 'Basis', 'leitfrage': 'Warum gibt es bei Butan mehrere räumliche Formen?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Was unterscheidet Butan von Isobutan?', 'antworten': ['Die C-Atome sind anders verknüpft: Butan ist unverzweigt, Isobutan verzweigt.', 'Butan und Isobutan unterscheiden sich in der Summenformel.', 'Isobutan enthält eine OH-Gruppe.', 'Verzweigung verändert keine physikalischen Eigenschaften.'], 'richtig': 0, 'feedback': 'Richtig: Es handelt sich um Konstitutionsisomerie mit anderer C-Gerüstform.', 'erklaerung': 'Butan kann um C-C-Einfachbindungen rotieren; verschiedene Formen haben verschiedene relative Energien.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'butan', 'typ': 'molecule', 'modul': 'Konformere', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum gibt es bei Butan mehrere räumliche Formen?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Was unterscheidet Butan von Isobutan?', 'mc_antworten': ['Die C-Atome sind anders verknüpft: Butan ist unverzweigt, Isobutan verzweigt.', 'Butan und Isobutan unterscheiden sich in der Summenformel.', 'Isobutan enthält eine OH-Gruppe.', 'Verzweigung verändert keine physikalischen Eigenschaften.'], 'mc_richtig': 0, 'feedback': 'Richtig: Es handelt sich um Konstitutionsisomerie mit anderer C-Gerüstform.', 'erklaerung': 'Butan kann um C-C-Einfachbindungen rotieren; verschiedene Formen haben verschiedene relative Energien.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': '1,2-Dichlorethan', 'smiles': 'ClCCCl', 'niveau': 'Basis', 'leitfrage': 'Wie beeinflusst Rotation die Orientierung der Chloratome?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Gruppen können im Raum unterschiedlich nahe kommen.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Gruppen können im Raum unterschiedlich nahe kommen.', 'erklaerung': 'Rotation verändert Abstände und Wechselwirkungen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': '1_2_dichlorethan', 'typ': 'molecule', 'modul': 'Konformere', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Wie beeinflusst Rotation die Orientierung der Chloratome?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Gruppen können im Raum unterschiedlich nahe kommen.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Gruppen können im Raum unterschiedlich nahe kommen.', 'erklaerung': 'Rotation verändert Abstände und Wechselwirkungen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Ethylenglykol', 'smiles': 'OCCO', 'niveau': 'Basis', 'leitfrage': 'Warum kann Ethylenglykol mehrere günstige Formen einnehmen?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Mehrere Einfachbindungen können rotieren.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Mehrere Einfachbindungen können rotieren.', 'erklaerung': 'Die beiden OH-Gruppen und flexible Bindungen ermöglichen mehrere Konformere.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'ethylenglykol', 'typ': 'molecule', 'modul': 'Konformere', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum kann Ethylenglykol mehrere günstige Formen einnehmen?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Mehrere Einfachbindungen können rotieren.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Mehrere Einfachbindungen können rotieren.', 'erklaerung': 'Die beiden OH-Gruppen und flexible Bindungen ermöglichen mehrere Konformere.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Cyclohexan', 'smiles': 'C1CCCCC1', 'niveau': 'Basis', 'leitfrage': 'Warum ist die Sesselstruktur von Cyclohexan wichtig?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Eine nicht-planare Form verringert ungünstige Spannungen.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Eine nicht-planare Form verringert ungünstige Spannungen.', 'erklaerung': 'Cyclohexan ist nicht planar; die Sesselstruktur ist energetisch günstig.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'cyclohexan', 'typ': 'molecule', 'modul': 'Konformere', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist die Sesselstruktur von Cyclohexan wichtig?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Eine nicht-planare Form verringert ungünstige Spannungen.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Eine nicht-planare Form verringert ungünstige Spannungen.', 'erklaerung': 'Cyclohexan ist nicht planar; die Sesselstruktur ist energetisch günstig.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Milchsäure', 'smiles': 'C[C@H](O)C(=O)O', 'niveau': 'Basis', 'leitfrage': 'Warum ist Milchsäure ein gutes Beispiel für Flexibilität und Chiralität?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Sie besitzt ein chirales Zentrum und mehrere polare Gruppen.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Sie besitzt ein chirales Zentrum und mehrere polare Gruppen.', 'erklaerung': 'Milchsäure verbindet funktionelle Gruppen, Chiralität und Beweglichkeit.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'milchsaeure', 'typ': 'molecule', 'modul': 'Konformere', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Milchsäure ein gutes Beispiel für Flexibilität und Chiralität?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Sie besitzt ein chirales Zentrum und mehrere polare Gruppen.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Sie besitzt ein chirales Zentrum und mehrere polare Gruppen.', 'erklaerung': 'Milchsäure verbindet funktionelle Gruppen, Chiralität und Beweglichkeit.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}], 'Wirkstoffe und Biomoleküle': [{'name': 'Coffein', 'smiles': 'Cn1cnc2n(C)c(=O)n(C)c(=O)c12', 'niveau': 'Basis', 'leitfrage': 'Warum kann Coffein mit Biomolekülen wechselwirken?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Coffein besitzt mehrere H-Brücken-Akzeptoren, aber keine Donoren.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Coffein besitzt mehrere H-Brücken-Akzeptoren, aber keine Donoren.', 'erklaerung': 'Das heteroatomreiche Ringsystem erlaubt mehrere schwache Wechselwirkungen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'coffein', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum kann Coffein mit Biomolekülen wechselwirken?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Coffein besitzt mehrere H-Brücken-Akzeptoren, aber keine Donoren.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Coffein besitzt mehrere H-Brücken-Akzeptoren, aber keine Donoren.', 'erklaerung': 'Das heteroatomreiche Ringsystem erlaubt mehrere schwache Wechselwirkungen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Aspirin', 'smiles': 'CC(=O)Oc1ccccc1C(=O)O', 'niveau': 'Basis', 'leitfrage': 'Welche funktionellen Gruppen prägen Aspirin?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Aspirin enthält Carboxylgruppe, Estergruppe und Aromat.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Aspirin enthält Carboxylgruppe, Estergruppe und Aromat.', 'erklaerung': 'Polare und unpolare Bereiche kommen in einem Wirkstoffmolekül zusammen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'aspirin', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Welche funktionellen Gruppen prägen Aspirin?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Aspirin enthält Carboxylgruppe, Estergruppe und Aromat.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Aspirin enthält Carboxylgruppe, Estergruppe und Aromat.', 'erklaerung': 'Polare und unpolare Bereiche kommen in einem Wirkstoffmolekül zusammen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Paracetamol', 'smiles': 'CC(=O)Nc1ccc(O)cc1', 'niveau': 'Basis', 'leitfrage': 'Welche Strukturmerkmale machen Paracetamol polar?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Paracetamol enthält Phenol und Amid.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Paracetamol enthält Phenol und Amid.', 'erklaerung': 'Diese Gruppen können H-Brücken bilden; der Aromat ist unpolarer.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'paracetamol', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Welche Strukturmerkmale machen Paracetamol polar?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Paracetamol enthält Phenol und Amid.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Paracetamol enthält Phenol und Amid.', 'erklaerung': 'Diese Gruppen können H-Brücken bilden; der Aromat ist unpolarer.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Ibuprofen', 'smiles': 'CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O', 'niveau': 'Basis', 'leitfrage': 'Warum ist Ibuprofen relativ lipophil?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Es besitzt eine polare Carboxylgruppe und einen großen unpolaren Bereich.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Es besitzt eine polare Carboxylgruppe und einen großen unpolaren Bereich.', 'erklaerung': 'Wirkstoffe enthalten oft beide Bereiche: polar für Wechselwirkungen, unpolar für Bindungstaschen/Membranen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'ibuprofen', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Ibuprofen relativ lipophil?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Es besitzt eine polare Carboxylgruppe und einen großen unpolaren Bereich.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Es besitzt eine polare Carboxylgruppe und einen großen unpolaren Bereich.', 'erklaerung': 'Wirkstoffe enthalten oft beide Bereiche: polar für Wechselwirkungen, unpolar für Bindungstaschen/Membranen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Acetazolamid', 'smiles': 'CC(=O)Nc1nnc(s1)S(N)(=O)=O', 'niveau': 'Basis', 'leitfrage': 'Warum ist Acetazolamid ein Brückenbeispiel zum Proteinlabor?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Es besitzt mehrere polare Gruppen und Heteroatome.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Es besitzt mehrere polare Gruppen und Heteroatome.', 'erklaerung': 'Als Carboanhydrase-Hemmstoff eignet es sich für Struktur-Wirkungs- und Protein-Ligand-Bezüge.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'acetazolamid', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Acetazolamid ein Brückenbeispiel zum Proteinlabor?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Es besitzt mehrere polare Gruppen und Heteroatome.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Es besitzt mehrere polare Gruppen und Heteroatome.', 'erklaerung': 'Als Carboanhydrase-Hemmstoff eignet es sich für Struktur-Wirkungs- und Protein-Ligand-Bezüge.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Glycin', 'smiles': 'NCC(=O)O', 'niveau': 'Basis', 'leitfrage': 'Warum ist Glycin die einfachste Aminosäure?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Glycin besitzt Aminogruppe und Carboxylgruppe.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Glycin besitzt Aminogruppe und Carboxylgruppe.', 'erklaerung': 'Die Seitenkette ist nur ein Wasserstoffatom.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'glycin', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Glycin die einfachste Aminosäure?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Glycin besitzt Aminogruppe und Carboxylgruppe.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Glycin besitzt Aminogruppe und Carboxylgruppe.', 'erklaerung': 'Die Seitenkette ist nur ein Wasserstoffatom.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Alanin', 'smiles': 'C[C@H](N)C(=O)O', 'niveau': 'Basis', 'leitfrage': 'Warum ist Alanin chiral?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Das zentrale C-Atom trägt vier verschiedene Gruppen.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Das zentrale C-Atom trägt vier verschiedene Gruppen.', 'erklaerung': 'Alanin ist ein einfaches Beispiel für Chiralität in Biomolekülen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'alanin', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Alanin chiral?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Das zentrale C-Atom trägt vier verschiedene Gruppen.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Das zentrale C-Atom trägt vier verschiedene Gruppen.', 'erklaerung': 'Alanin ist ein einfaches Beispiel für Chiralität in Biomolekülen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Phenylalanin', 'smiles': 'N[C@@H](Cc1ccccc1)C(=O)O', 'niveau': 'Basis', 'leitfrage': 'Was unterscheidet Phenylalanin von Glycin und Alanin?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Die Seitenkette enthält einen aromatischen Ring.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Die Seitenkette enthält einen aromatischen Ring.', 'erklaerung': 'Aromatische hydrophobe Seitenketten sind wichtig für Proteinstruktur und Wechselwirkungen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'phenylalanin', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Was unterscheidet Phenylalanin von Glycin und Alanin?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Die Seitenkette enthält einen aromatischen Ring.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Die Seitenkette enthält einen aromatischen Ring.', 'erklaerung': 'Aromatische hydrophobe Seitenketten sind wichtig für Proteinstruktur und Wechselwirkungen.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Adenin', 'smiles': 'Nc1ncnc2ncnc12', 'niveau': 'Basis', 'leitfrage': 'Warum ist Adenin als Nukleobase interessant?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Adenin ist eine stickstoffhaltige Nukleobase.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Adenin ist eine stickstoffhaltige Nukleobase.', 'erklaerung': 'Das heteroaromatische Ringsystem kann an Basenpaarung beteiligt sein.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'adenin', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Adenin als Nukleobase interessant?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Adenin ist eine stickstoffhaltige Nukleobase.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Adenin ist eine stickstoffhaltige Nukleobase.', 'erklaerung': 'Das heteroaromatische Ringsystem kann an Basenpaarung beteiligt sein.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Ribose', 'smiles': 'O=CC(O)C(O)C(O)CO', 'niveau': 'Basis', 'leitfrage': 'Warum ist Ribose ein wichtiger RNA-Baustein?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'frage': 'Welche Aussage passt am besten?', 'antworten': ['Ribose besitzt mehrere OH-Gruppen und ist stark polar.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'richtig': 0, 'feedback': 'Richtig: Ribose besitzt mehrere OH-Gruppen und ist stark polar.', 'erklaerung': 'In RNA ist Ribose Teil des Zucker-Phosphat-Rückgrats; hier wird die offenkettige Form vereinfacht gezeigt.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'id': 'ribose', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Ribose ein wichtiger RNA-Baustein?', 'beobachten': ['Betrachte zuerst die 2D-Struktur.', 'Drehe anschließend das 3D-Modell.', 'Vergleiche die Deskriptoren mit der Leitfrage.'], 'mc_frage': 'Welche Aussage passt am besten?', 'mc_antworten': ['Ribose besitzt mehrere OH-Gruppen und ist stark polar.', 'Die Summenformel allein erklärt alle Stoffeigenschaften.', 'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.', 'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'], 'mc_richtig': 0, 'feedback': 'Richtig: Ribose besitzt mehrere OH-Gruppen und ist stark polar.', 'erklaerung': 'In RNA ist Ribose Teil des Zucker-Phosphat-Rückgrats; hier wird die offenkettige Form vereinfacht gezeigt.', 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, {'name': 'Citronensäure', 'smiles': 'O=C(O)CC(O)(CC(=O)O)C(=O)O', 'niveau': 'Vertiefung', 'leitfrage': 'Warum ist Citronensäure stark wasserfreundlich und als Stoffwechselprodukt interessant?', 'beobachten': ['Suche die drei Carboxylgruppen.', 'Suche die OH-Gruppe.', 'Vergleiche TPSA und H-Brücken-Möglichkeiten.'], 'frage': 'Warum ist Citronensäure stark wasserfreundlich und reaktiv in Säure-Base-Zusammenhängen?', 'antworten': ['Mehrere Carboxylgruppen und eine OH-Gruppe ermöglichen starke Wechselwirkungen mit Wasser.', 'Citronensäure besteht fast nur aus unpolaren C-H-Bindungen.', 'Citronensäure ist ein Protein und deshalb wasserlöslich.', 'Nur die Molmasse entscheidet über Wasserlöslichkeit.'], 'richtig': 0, 'feedback': 'Richtig: Mehrere polare Gruppen prägen die Wasserfreundlichkeit und Säureeigenschaften.', 'erklaerung': 'Citronensäure ist noch überschaubar, zeigt aber bereits sehr deutlich, wie mehrere polare funktionelle Gruppen ein Molekül stark wasserfreundlich machen. Gleichzeitig stellt sie eine Brücke zum Stoffwechsel dar.', 'modellgrenze': 'In biologischen Systemen liegen Carbonsäuren häufig teilweise oder vollständig deprotoniert vor; diese einfache Darstellung bildet pH- und Salzformen nicht vollständig ab.', 'id': 'citronensaeure', 'typ': 'molecule', 'modul': 'Wirkstoffe und Biomoleküle', 'tags': [], 'didaktik': {'fehlvorstellung': None, 'leitfrage': 'Warum ist Citronensäure stark wasserfreundlich und als Stoffwechselprodukt interessant?', 'beobachten': ['Suche die drei Carboxylgruppen.', 'Suche die OH-Gruppe.', 'Vergleiche TPSA und H-Brücken-Möglichkeiten.'], 'mc_frage': 'Warum ist Citronensäure stark wasserfreundlich und reaktiv in Säure-Base-Zusammenhängen?', 'mc_antworten': ['Mehrere Carboxylgruppen und eine OH-Gruppe ermöglichen starke Wechselwirkungen mit Wasser.', 'Citronensäure besteht fast nur aus unpolaren C-H-Bindungen.', 'Citronensäure ist ein Protein und deshalb wasserlöslich.', 'Nur die Molmasse entscheidet über Wasserlöslichkeit.'], 'mc_richtig': 0, 'feedback': 'Richtig: Mehrere polare Gruppen prägen die Wasserfreundlichkeit und Säureeigenschaften.', 'erklaerung': 'Citronensäure ist noch überschaubar, zeigt aber bereits sehr deutlich, wie mehrere polare funktionelle Gruppen ein Molekül stark wasserfreundlich machen. Gleichzeitig stellt sie eine Brücke zum Stoffwechsel dar.', 'modellgrenze': 'In biologischen Systemen liegen Carbonsäuren häufig teilweise oder vollständig deprotoniert vor; diese einfache Darstellung bildet pH- und Salzformen nicht vollständig ab.', 'mini_aufgaben': [], 'experimenteller_bezug': None}, 'stoffdaten': {'schmelzpunkt_C': None, 'siedepunkt_C': None, 'dichte_g_cm3': None, 'wasserloeslichkeit': None, 'aggregatzustand_raumtemperatur': None, 'bedingungen': None, 'quelle': None, 'hinweis': None}, 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}]}, 'isomer_pairs': {'Ethanol': 'Dimethylether', 'Dimethylether': 'Ethanol', 'Propanal': 'Aceton', 'Aceton': 'Propanal', 'Butan': 'Isobutan', 'Isobutan': 'Butan', 'Propansäure': 'Essigsäuremethylester', 'Essigsäuremethylester': 'Propansäure', 'D-Alanin': 'L-Alanin', 'L-Alanin': 'D-Alanin', 'Glucose offen': 'Fructose offen', 'Fructose offen': 'Glucose offen', 'Maleinsäure': 'Fumarsäure', 'Fumarsäure': 'Maleinsäure'}, 'polarity_pairs': {'Wasser vs. Schwefelwasserstoff': {'a': 'Wasser', 'b': 'Schwefelwasserstoff', 'fehlvorstellung': 'Ähnliche Summenformel und ähnliche Molekülform bedeuten ähnliche Eigenschaften.', 'leitfrage': 'Warum ist Wasser bei Raumtemperatur flüssig, Schwefelwasserstoff aber gasförmig?', 'mini_aufgaben': ['Vergleiche die Molekülform beider Moleküle.', 'Suche mögliche Wasserstoffbrücken.', 'Erkläre, warum Wasser ein starkes Netzwerk zwischen den Molekülen bilden kann.'], 'kernaussage': 'Wasser bildet starke Wasserstoffbrücken; Schwefelwasserstoff hat deutlich schwächere zwischenmolekulare Wechselwirkungen.', 'experiment': 'Wasser ist bei Raumtemperatur flüssig; Schwefelwasserstoff ist gasförmig.', 'modellgrenze': 'Für kleine anorganische Moleküle sind logP/TPSA nur eingeschränkt aussagekräftig. Hier zählt der experimentelle Bezug besonders stark.', 'id': 'wasser_vs_schwefelwasserstoff', 'typ': 'comparison', 'modul': 'Polarität und Löslichkeit', 'experimenteller_bezug': 'Wasser ist bei Raumtemperatur flüssig; Schwefelwasserstoff ist gasförmig.', 'stoffdaten_hinweis': 'Stoffdaten sind als kuratierte Unterrichtswerte bei den Einzelmolekülen hinterlegt, wo sie didaktisch besonders hilfreich sind.', 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, 'Methan vs. Hexan': {'a': 'Methan', 'b': 'Hexan', 'fehlvorstellung': 'Unpolar ist unpolar – daher müssten unpolare Stoffe ähnliche Eigenschaften besitzen.', 'leitfrage': 'Warum ist Methan gasförmig, Hexan aber flüssig, obwohl beide unpolar sind?', 'mini_aufgaben': ['Vergleiche die Molekülgröße.', 'Betrachte die unpolare Oberfläche.', 'Erkläre den Einfluss der Molekülgröße auf London-Dispersionskräfte.'], 'kernaussage': 'Auch unpolare Moleküle ziehen einander an. Größere Moleküle besitzen stärkere London-Kräfte.', 'experiment': 'Methan ist gasförmig; Hexan ist flüssig und kaum wasserlöslich.', 'modellgrenze': 'London-Kräfte werden nicht direkt berechnet, sondern nur über Größe und Oberfläche erschlossen.', 'id': 'methan_vs_hexan', 'typ': 'comparison', 'modul': 'Polarität und Löslichkeit', 'experimenteller_bezug': 'Methan ist gasförmig; Hexan ist flüssig und kaum wasserlöslich.', 'stoffdaten_hinweis': 'Stoffdaten sind als kuratierte Unterrichtswerte bei den Einzelmolekülen hinterlegt, wo sie didaktisch besonders hilfreich sind.', 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, 'Ethanol vs. 1-Hexanol': {'a': 'Ethanol', 'b': '1-Hexanol', 'fehlvorstellung': 'Alkohole sind wegen der OH-Gruppe immer gut wasserlöslich.', 'leitfrage': 'Warum ist Ethanol gut wasserlöslich, 1-Hexanol aber deutlich schlechter?', 'mini_aufgaben': ['Markiere in beiden Molekülen die OH-Gruppe.', 'Vergleiche die Größe der unpolaren Kohlenwasserstoffreste.', 'Begründe die unterschiedliche Wasserlöslichkeit.'], 'kernaussage': 'Die OH-Gruppe ermöglicht H-Brücken, aber bei längerer Kohlenstoffkette wird der unpolare Anteil immer wichtiger.', 'experiment': 'Ethanol ist mit Wasser mischbar; 1-Hexanol ist nur begrenzt wasserlöslich.', 'modellgrenze': 'Die Skala ist eine Orientierung und ersetzt keine Messung der Löslichkeit.', 'id': 'ethanol_vs_1_hexanol', 'typ': 'comparison', 'modul': 'Polarität und Löslichkeit', 'experimenteller_bezug': 'Ethanol ist mit Wasser mischbar; 1-Hexanol ist nur begrenzt wasserlöslich.', 'stoffdaten_hinweis': 'Stoffdaten sind als kuratierte Unterrichtswerte bei den Einzelmolekülen hinterlegt, wo sie didaktisch besonders hilfreich sind.', 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, 'Essigsäure vs. Palmitinsäure': {'a': 'Essigsäure', 'b': 'Palmitinsäure', 'fehlvorstellung': 'Carbonsäuren sind wegen der COOH-Gruppe grundsätzlich gut wasserlöslich.', 'leitfrage': 'Warum ist Essigsäure gut wasserlöslich, Palmitinsäure aber kaum?', 'mini_aufgaben': ['Suche die Carboxylgruppe in beiden Molekülen.', 'Vergleiche die unpolaren Molekülteile.', 'Erkläre, warum dieselbe funktionelle Gruppe nicht dieselbe Stoffeigenschaft garantiert.'], 'kernaussage': 'Eine polare Gruppe kann durch einen sehr großen unpolaren Rest in ihrer Wirkung überlagert werden.', 'experiment': 'Essigsäure ist gut wasserlöslich; Palmitinsäure ist in Wasser praktisch unlöslich.', 'modellgrenze': 'pH-Effekte, Salzbildung, Kristallstruktur und Aggregation werden nicht abgebildet.', 'id': 'essigsaeure_vs_palmitinsaeure', 'typ': 'comparison', 'modul': 'Polarität und Löslichkeit', 'experimenteller_bezug': 'Essigsäure ist gut wasserlöslich; Palmitinsäure ist in Wasser praktisch unlöslich.', 'stoffdaten_hinweis': 'Stoffdaten sind als kuratierte Unterrichtswerte bei den Einzelmolekülen hinterlegt, wo sie didaktisch besonders hilfreich sind.', 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, 'Diethylether vs. Diethylenglycol': {'a': 'Diethylether', 'b': 'Diethylenglycol', 'fehlvorstellung': 'Ein Sauerstoffatom reicht aus, um Wasserlöslichkeit zuverlässig vorherzusagen.', 'leitfrage': 'Warum ist Diethylenglycol deutlich wasserfreundlicher als Diethylether?', 'mini_aufgaben': ['Zähle die O-Atome.', 'Suche OH-Gruppen.', 'Vergleiche H-Brücken-Donoren und -Akzeptoren.'], 'kernaussage': 'Diethylether kann H-Brücken nur aufnehmen; Diethylenglycol kann durch OH-Gruppen auch spenden und besitzt eine größere polare Oberfläche.', 'experiment': 'Diethylether ist nur begrenzt wasserlöslich; Diethylenglycol ist sehr wasserfreundlich.', 'modellgrenze': 'Toxizität und experimentelle Verwendung werden hier nicht behandelt.', 'id': 'diethylether_vs_diethylenglycol', 'typ': 'comparison', 'modul': 'Polarität und Löslichkeit', 'experimenteller_bezug': 'Diethylether ist nur begrenzt wasserlöslich; Diethylenglycol ist sehr wasserfreundlich.', 'stoffdaten_hinweis': 'Stoffdaten sind als kuratierte Unterrichtswerte bei den Einzelmolekülen hinterlegt, wo sie didaktisch besonders hilfreich sind.', 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, 'Methanol vs. Methanal': {'a': 'Methanol', 'b': 'Methanal', 'fehlvorstellung': 'Wenn beide Moleküle ein O-Atom besitzen, müssen sie ähnlich mit Wasser wechselwirken.', 'leitfrage': 'Warum unterscheiden sich Methanol und Methanal trotz ähnlicher Größe in ihren Wechselwirkungen?', 'mini_aufgaben': ['Suche OH-Gruppe bzw. Carbonylgruppe.', 'Vergleiche H-Brücken-Donoren und -Akzeptoren.', 'Erkläre den Unterschied zwischen Alkohol und Aldehyd.'], 'kernaussage': 'Methanol kann H-Brücken spenden und aufnehmen; Methanal kann nur aufnehmen und ist als Carbonylverbindung anders reaktiv.', 'experiment': 'Beide sind klein und wasserfreundlich, aber chemisch deutlich verschieden.', 'modellgrenze': 'Methanal liegt in Wasser teilweise hydratisiert vor; das einfache Modell bildet dies nicht ab.', 'id': 'methanol_vs_methanal', 'typ': 'comparison', 'modul': 'Polarität und Löslichkeit', 'experimenteller_bezug': 'Beide sind klein und wasserfreundlich, aber chemisch deutlich verschieden.', 'stoffdaten_hinweis': 'Stoffdaten sind als kuratierte Unterrichtswerte bei den Einzelmolekülen hinterlegt, wo sie didaktisch besonders hilfreich sind.', 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}, 'Aceton vs. Essigsäure': {'a': 'Aceton', 'b': 'Essigsäure', 'fehlvorstellung': 'Eine C=O-Gruppe bestimmt die Eigenschaften allein.', 'leitfrage': 'Warum verhalten sich Aceton und Essigsäure trotz Carbonylgruppe unterschiedlich?', 'mini_aufgaben': ['Suche die Carbonylgruppe.', 'Suche zusätzliche funktionelle Gruppen.', 'Vergleiche H-Brücken und Säureeigenschaft.'], 'kernaussage': 'Aceton ist polar aprotisch; Essigsäure besitzt zusätzlich eine OH-Gruppe, kann H-Brücken spenden und ist sauer.', 'experiment': 'Beide sind gut wasserlöslich/mischbar, aber Essigsäure zeigt Säure-Base-Verhalten und starke H-Brücken.', 'modellgrenze': 'pH-Gleichgewichte und vollständige Säuredissoziation werden nicht berechnet.', 'id': 'aceton_vs_essigsaeure', 'typ': 'comparison', 'modul': 'Polarität und Löslichkeit', 'experimenteller_bezug': 'Beide sind gut wasserlöslich/mischbar, aber Essigsäure zeigt Säure-Base-Verhalten und starke H-Brücken.', 'stoffdaten_hinweis': 'Stoffdaten sind als kuratierte Unterrichtswerte bei den Einzelmolekülen hinterlegt, wo sie didaktisch besonders hilfreich sind.', 'reaktivitaet': {'hinweis': None, 'warum': None, 'reaktionsidee': None, 'modellgrenze': None}}}, 'future_datasets': {'basis': None, 'standard': 'molecules_v0_4_0.json', 'plus': None}}
def load_teaching_data():
    """Lädt Unterrichtsdaten aus lokaler JSON, GitHub Raw oder Fallback.

    Diese Struktur bereitet spätere alternative JSON-Datenpakete vor. Für die App
    bleiben die Oberflächenelemente bewusst schlank.
    """
    import os, urllib.request
    candidates = [LOCAL_DATA_FILE, f"/content/{LOCAL_DATA_FILE}"]
    for path in candidates:
        if os.path.exists(path):
            try:
                with open(path, "r", encoding="utf-8") as f:
                    return json.load(f)
            except Exception as exc:
                print(f"Hinweis: Lokale Daten konnten nicht geladen werden: {exc}")
    try:
        with urllib.request.urlopen(DATA_URL, timeout=8) as response:
            return json.loads(response.read().decode("utf-8"))
    except Exception as exc:
        print("Hinweis: Nutze eingebettete Fallback-Daten, da die externe JSON nicht erreichbar ist.")
        return FALLBACK_DATA

DATA = load_teaching_data()
if "modules" in DATA:
    MOLECULES = DATA["modules"]
    ISOMER_PAIRS = DATA.get("isomer_pairs", {})
    POLARITY_PAIRS = DATA.get("polarity_pairs", {})
else:
    # Abwärtskompatibilität zu älteren JSON-Dateien.
    MOLECULES = {k:v for k,v in DATA.items() if isinstance(v, list)}
    ISOMER_PAIRS = {}
    POLARITY_PAIRS = {}


FUNCTIONAL_GROUPS = {
    "Alkohol": "[OX2H][CX4]",
    "Phenol": "[OX2H]c",
    "Aldehyd": "[CX3H1](=O)[#6,#1]",
    "Keton": "[#6][CX3](=O)[#6]",
    "Carbonsäure": "C(=O)[OX2H1]",
    "Ester": "C(=O)O[#6]",
    "Ether": "[#6][OD2][#6]",
    "Carbonsäureanhydrid": "C(=O)OC(=O)",
    "Amin": "[NX3;H2,H1,H0;!$(NC=O)]",
    "Amid": "C(=O)N",
    "Aromatischer Ring": "a",
    "Carbonylgruppe": "[CX3]=[OX1]",
}

current_report = {}

def mol_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Ungültiger SMILES-Code.")
    return mol

def make_3d(mol):
    mol_h = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    status = AllChem.EmbedMolecule(mol_h, params)
    if status != 0:
        AllChem.EmbedMolecule(mol_h, randomSeed=42)
    try:
        AllChem.MMFFOptimizeMolecule(mol_h, maxIters=300)
    except Exception:
        try:
            AllChem.UFFOptimizeMolecule(mol_h, maxIters=300)
        except Exception:
            pass
    return mol_h

def calc_properties(mol):
    return {
        "Summenformel": rdMolDescriptors.CalcMolFormula(mol),
        "Molmasse / g mol⁻¹": round(Descriptors.MolWt(mol), 2),
        "H-Brücken-Donoren": rdMolDescriptors.CalcNumHBD(mol),
        "H-Brücken-Akzeptoren": rdMolDescriptors.CalcNumHBA(mol),
        "logP (Modellwert)": round(Crippen.MolLogP(mol), 2),
        "TPSA / Å²": round(rdMolDescriptors.CalcTPSA(mol), 2),
        "rotierbare Bindungen": rdMolDescriptors.CalcNumRotatableBonds(mol),
        "Ringanzahl": rdMolDescriptors.CalcNumRings(mol),
        "schwere Atome": mol.GetNumHeavyAtoms(),
    }

def detect_functional_groups(mol):
    found=[]
    for name, smarts in FUNCTIONAL_GROUPS.items():
        patt = Chem.MolFromSmarts(smarts)
        if patt is not None and mol.HasSubstructMatch(patt):
            found.append(name)
    return sorted(set(found))

def mol_png_data_uri(mol, width=340, height=260):
    """2D-Struktur als eingebettetes PNG für kompakte HTML-Layouts."""
    img = Draw.MolToImage(mol, size=(width, height))
    buf = BytesIO()
    img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    return f"data:image/png;base64,{b64}"

def show_2d(mol, width=360, height=260):
    display(HTML(f"<img src='{mol_png_data_uri(mol, width, height)}' style='max-width:100%;'>"))

def _style_for_3d(style_name):
    if style_name == "Kalottenmodell":
        return {"sphere": {"scale": 1.0}}
    if style_name == "Stabmodell":
        return {"stick": {"radius": 0.16}}
    # Default: Kugel-Stab
    return {"stick": {"radius": 0.12}, "sphere": {"scale": 0.25}}

def _style_js_for_3d(style_name):
    if style_name == "Kalottenmodell":
        return '{sphere:{scale:1.0}}'
    if style_name == "Stabmodell":
        return '{stick:{radius:0.16}}'
    if style_name == "Didaktische Polaritätsoberfläche":
        # Die Oberfläche selbst wird unten zusätzlich erzeugt.
        return '{stick:{radius:0.08}, sphere:{scale:0.18}}'
    return '{stick:{radius:0.12}, sphere:{scale:0.25}}'

def _is_polar_surface(style_name):
    return style_name == "Didaktische Polaritätsoberfläche"

def _custom_3dmol_iframe_html(molblock, width=520, height=380, style="Kugel-Stab"):
    # Schlanke 3Dmol.js-Ausgabe ohne py3Dmol-Wrapper.
    # v0.2: optional mit didaktischer Polaritätsoberfläche.
    div_id = "mol3d_" + uuid.uuid4().hex
    mol_js = json.dumps(molblock)
    style_js = _style_js_for_3d(style)
    add_surface_js = ""
    if _is_polar_surface(style):
        add_surface_js = """
      try {
        // Didaktische Oberfläche: elementbasierte VDW-Oberfläche, keine exakte ESP-Fläche.
        viewer.addSurface($3Dmol.SurfaceType.VDW, {opacity:0.72, colorscheme:'Jmol'}, {});
      } catch(surfaceErr) {
        console.log('Oberfläche konnte nicht erzeugt werden', surfaceErr);
      }
        """
    inner = f'''
<!doctype html>
<html>
<head>
<meta charset="utf-8">
<style>
  html, body {{ margin:0; padding:0; width:100%; height:100%; overflow:hidden; background:white; }}
  #{div_id} {{ width:100%; height:100%; }}
  .msg {{ font-family:Arial, sans-serif; font-size:12px; padding:8px; color:#666; }}
</style>
<script src="https://cdn.jsdelivr.net/npm/3dmol@2.5.5/build/3Dmol-min.js"></script>
</head>
<body>
<div id="{div_id}"><div class="msg">3D-Modell wird geladen …</div></div>
<script>
(function() {{
  const molblock = {mol_js};
  const styleObj = {style_js};
  function init(attempt) {{
    const el = document.getElementById('{div_id}');
    if (!el) return;
    if (typeof $3Dmol === 'undefined') {{
      if (attempt < 20) {{ window.setTimeout(function() {{ init(attempt+1); }}, 150); }}
      else {{ el.innerHTML = '<div class="msg">3Dmol.js konnte nicht geladen werden. Bitte erneut analysieren oder 3D-Engine wechseln.</div>'; }}
      return;
    }}
    el.innerHTML = '';
    try {{
      const viewer = $3Dmol.createViewer(el, {{backgroundColor:'white'}});
      viewer.addModel(molblock, 'sdf');
      viewer.setStyle({{}}, styleObj);
      {add_surface_js}
      viewer.zoomTo();
      viewer.render();
      window.addEventListener('resize', function() {{ viewer.resize(); viewer.render(); }});
    }} catch(e) {{
      el.innerHTML = '<div class="msg">3D-Anzeige fehlgeschlagen: ' + e + '</div>';
    }}
  }}
  init(0);
}})();
</script>
</body>
</html>
'''
    escaped = html.escape(inner, quote=True)
    return f'<iframe class="mol-lab-3d-frame" srcdoc="{escaped}" width="{width}" height="{height}" loading="eager" style="border:1px solid #ddd; border-radius:8px; background:white; max-width:100%;"></iframe>'

def _py3dmol_iframe_html(viewer, width=520, height=380):
    # Fallback-Ausgabe mit py3Dmol-eigenem HTML im iframe.
    try:
        inner = viewer._make_html()
        escaped = html.escape(inner, quote=True)
        return f'<iframe class="mol-lab-3d-frame" srcdoc="{escaped}" width="{width}" height="{height}" loading="eager" style="border:1px solid #ddd; border-radius:8px; background:white; max-width:100%;"></iframe>'
    except Exception as e:
        return f"<p style='color:#b00020'>3D-Anzeige konnte nicht erzeugt werden: {html.escape(str(e))}</p>"

def _viewer_for_molblock(molblock, width=520, height=380, style="Kugel-Stab"):
    viewer = py3Dmol.view(width=width, height=height)
    viewer.addModel(molblock, "sdf")
    viewer.setStyle(_style_for_3d(style))
    if _is_polar_surface(style):
        try:
            viewer.addSurface(py3Dmol.VDW, {"opacity":0.72, "colorscheme":"Jmol"})
        except Exception:
            pass
    viewer.setBackgroundColor("white")
    viewer.zoomTo()
    return viewer

def molblock_3d_html(molblock, width=520, height=380, style="Kugel-Stab", mode="custom"):
    if mode == "py3dmol":
        viewer = _viewer_for_molblock(molblock, width=width, height=height, style=style)
        return _py3dmol_iframe_html(viewer, width=width+20, height=height+30)
    return _custom_3dmol_iframe_html(molblock, width=width+20, height=height+30, style=style)

def mol3d_iframe_html(mol3d, width=520, height=380, style="Kugel-Stab", mode="custom"):
    mb = Chem.MolToMolBlock(mol3d)
    return molblock_3d_html(mb, width=width, height=height, style=style, mode=mode)

def show_3d(mol3d, width=520, height=380, style="Kugel-Stab", mode="custom"):
    display(HTML(mol3d_iframe_html(mol3d, width=width, height=height, style=style, mode=mode)))


def _label_bar_html(labels, active_index):
    """Kleine qualitative Skala als HTML."""
    cells=[]
    for i,lab in enumerate(labels):
        if i == active_index:
            cells.append(f"<span style='display:inline-block; padding:3px 7px; margin:1px; border-radius:10px; background:#e8f0fe; border:1px solid #5f7fcf; font-weight:bold;'>{html.escape(lab)}</span>")
        else:
            cells.append(f"<span style='display:inline-block; padding:3px 7px; margin:1px; border-radius:10px; background:#f6f6f6; border:1px solid #ddd; color:#666;'>{html.escape(lab)}</span>")
    return " ".join(cells)

def _formal_charge(mol):
    try:
        return sum(atom.GetFormalCharge() for atom in mol.GetAtoms())
    except Exception:
        return 0

def _hetero_heavy_count(mol):
    return sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() not in (1,6))

def qualitative_polarity_assessment(mol, props):
    """Didaktische Näherung für Polarität und Wasserlöslichkeit.

    Absichtlich keine exakte Stoffdatenvorhersage: Die Skalen übersetzen einfache
    RDKit-Modellgrößen in eine schülernahe Orientierung.
    """
    logp = float(props.get("logP (Modellwert)", 0) or 0)
    tpsa = float(props.get("TPSA / Å²", 0) or 0)
    hbd = int(props.get("H-Brücken-Donoren", 0) or 0)
    hba = int(props.get("H-Brücken-Akzeptoren", 0) or 0)
    heavy = max(int(props.get("schwere Atome", 1) or 1), 1)
    charge = abs(_formal_charge(mol))
    hetero = _hetero_heavy_count(mol)

    # Polaritätseindruck: eher Dichte polarer Oberfläche + H-Brücken-Möglichkeiten,
    # abgeschwächt durch deutlich lipophile Anteile.
    polar_score = 0.0
    polar_score += min(tpsa / 25.0, 3.0)
    polar_score += min(hbd * 0.8, 1.6)
    polar_score += min(hba * 0.25, 1.0)
    polar_score += min(charge * 1.4, 2.8)
    polar_score -= max(logp, 0) * 0.45
    polar_score -= max((heavy - 2*max(hetero,1)) / 18.0, 0)
    polar_score = max(polar_score, 0)

    if polar_score < 0.4:
        pol_i = 0
    elif polar_score < 1.0:
        pol_i = 1
    elif polar_score < 1.75:
        pol_i = 2
    elif polar_score < 2.8:
        pol_i = 3
    else:
        pol_i = 4

    # Wasserlöslichkeits-Tendenz: H-Brücken und TPSA begünstigen, logP und Größe
    # erschweren. Kleine polare Carbonyl-/Heteroatom-Moleküle erhalten einen Bonus.
    sol_score = 0.0
    sol_score += hbd * 1.2 + hba * 1.1 + tpsa / 25.0
    sol_score += min(charge * 3.0, 4.0)
    sol_score -= max(logp, 0) * 1.15
    sol_score -= max(heavy - 6, 0) * 0.25
    if heavy <= 5 and hba >= 1 and tpsa > 14:
        sol_score += 0.75
    sol_score = max(sol_score, 0)

    if sol_score < 0.45:
        sol_i = 0
    elif sol_score < 1.45:
        sol_i = 1
    elif sol_score < 2.35:
        sol_i = 2
    elif sol_score < 3.5:
        sol_i = 3
    else:
        sol_i = 4

    pol_labels = ["sehr schwach", "schwach", "mittel", "stark", "sehr stark"]
    sol_labels = ["kaum wasserlöslich", "begrenzt", "mäßig", "gut", "sehr gut"]

    reasons=[]
    if hbd > 0:
        reasons.append("kann H-Brücken spenden")
    if hba > 0:
        reasons.append("kann H-Brücken aufnehmen")
    if tpsa >= 70:
        reasons.append("große polare Oberfläche")
    elif tpsa >= 20:
        reasons.append("erkennbare polare Oberfläche")
    if logp >= 2.5:
        reasons.append("deutlich lipophiler Anteil")
    elif logp >= 0.8:
        reasons.append("unpolarer/lipophiler Anteil nimmt zu")
    if charge:
        reasons.append("formale Ladung vorhanden")
    if not reasons:
        reasons.append("kaum polare Gruppen erkennbar")

    return {
        "polarity_index": pol_i,
        "polarity_label": pol_labels[pol_i],
        "solubility_index": sol_i,
        "solubility_label": sol_labels[sol_i],
        "polarity_score": round(polar_score, 2),
        "solubility_score": round(sol_score, 2),
        "reason": "; ".join(reasons) + "."
    }

def _polarity_panel_html(mol, props):
    a = qualitative_polarity_assessment(mol, props)
    pol_labels = ["sehr schwach", "schwach", "mittel", "stark", "sehr stark"]
    sol_labels = ["kaum wasserlöslich", "begrenzt", "mäßig", "gut", "sehr gut"]
    return f"""
    <div style='margin-top:8px; padding:8px; border:1px solid #e0e0e0; border-radius:8px; background:#fbfbfb;'>
      <div style='font-size:0.95em; margin-bottom:5px;'><b>Didaktischer Polaritätseindruck:</b> {html.escape(a['polarity_label'])}</div>
      <div style='font-size:0.82em; margin-bottom:8px;'>{_label_bar_html(pol_labels, a['polarity_index'])}</div>
      <div style='font-size:0.95em; margin-bottom:5px;'><b>Wasserlöslichkeits-Tendenz:</b> {html.escape(a['solubility_label'])}</div>
      <div style='font-size:0.82em; margin-bottom:8px;'>{_label_bar_html(sol_labels, a['solubility_index'])}</div>
      <div style='font-size:0.82em; color:#555;'><b>Kurzbegründung:</b> {html.escape(a['reason'])}</div>
      <details style='font-size:0.78em; color:#555; line-height:1.32; margin-top:6px;'>
        <summary style='cursor:pointer;'>Näherung und Modellgrenzen anzeigen</summary>
        <div style='margin-top:5px;'>Diese Einschätzungen sind didaktische Näherungen. Sie werden aus einfachen Modellwerten wie logP, TPSA, Wasserstoffbrücken-Donoren/-Akzeptoren, Molekülgröße und formaler Ladung abgeleitet. Sie sollen Trends verständlich machen, ersetzen aber keine experimentellen Löslichkeitsdaten. Die tatsächliche Wasserlöslichkeit hängt zusätzlich von Temperatur, Ionisierungszustand, Konzentration, Molekülform und Wechselwirkungen mit dem Lösungsmittel ab.</div>
      </details>
    </div>
    """

def _surface_legend_html(style_name):
    if not _is_polar_surface(style_name):
        return ""
    return """
    <div style='font-size:0.78em; color:#555; line-height:1.3; margin-top:5px;'>
      <b>Didaktische Oberfläche:</b> elementbasierte Näherung, keine exakte elektrostatische Potentialfläche.
      Rot/blau/gelb/grün markieren heteroatomreiche, häufig polarere Bereiche; grau/weiß überwiegend C/H-Bereiche.
    </div>
    """

def _props_table_html(props):
    rows = []
    for k, v in props.items():
        rows.append(f"<tr><td style='padding:4px 8px; border-bottom:1px solid #eee;'>{html.escape(str(k))}</td><td style='padding:4px 8px; border-bottom:1px solid #eee; text-align:right;'><b>{html.escape(str(v))}</b></td></tr>")
    return "<table style='border-collapse:collapse; width:100%; font-size:0.92em;'>" + "".join(rows) + "</table>"

def _descriptor_note_html():
    return """
    <div style='font-size:0.78em; color:#555; line-height:1.32; margin-top:8px; border-top:1px solid #eee; padding-top:6px;'>
      <b>logP:</b> Modellwert für die Verteilung zwischen Fettphase und Wasser. Größer = eher lipophil, kleiner/negativ = eher hydrophil.<br>
      <b>TPSA:</b> topologische polare Oberfläche. Größer = mehr polare Oberfläche, oft stärkere Wechselwirkung mit Wasser/H-Brücken.
    </div>
    """


def _safe_text(x):
    return "" if x is None else str(x)

def _has_stoffdaten(card):
    sd = card.get("stoffdaten") or {}
    return any(sd.get(k) not in (None, "", []) for k in ["schmelzpunkt_C", "siedepunkt_C", "dichte_g_cm3", "wasserloeslichkeit", "aggregatzustand_raumtemperatur", "hinweis"])

def _stoffdaten_compact_html(card):
    sd = card.get("stoffdaten") or {}
    if not _has_stoffdaten(card):
        return ""
    rows=[]
    labels=[
        ("Aggregatzustand", "aggregatzustand_raumtemperatur"),
        ("Schmelzpunkt / °C", "schmelzpunkt_C"),
        ("Siedepunkt / °C", "siedepunkt_C"),
        ("Dichte / g cm⁻³", "dichte_g_cm3"),
        ("Wasserlöslichkeit", "wasserloeslichkeit"),
    ]
    for label,key in labels:
        val = sd.get(key)
        if val not in (None, "", []):
            rows.append(f"<tr><td style='padding:3px 7px;border-bottom:1px solid #eee;'>{html.escape(label)}</td><td style='padding:3px 7px;border-bottom:1px solid #eee;'><b>{html.escape(str(val))}</b></td></tr>")
    if sd.get("hinweis"):
        rows.append(f"<tr><td style='padding:3px 7px;border-bottom:1px solid #eee;'>Hinweis</td><td style='padding:3px 7px;border-bottom:1px solid #eee;'>{html.escape(str(sd.get('hinweis')))}</td></tr>")
    if not rows:
        return ""
    source = sd.get("quelle") or "kuratierte Unterrichtsdaten"
    return "<table style='border-collapse:collapse;width:100%;font-size:0.88em;'>" + "".join(rows) + "</table>" + f"<div style='font-size:0.75em;color:#666;margin-top:4px;'>Quelle/Einordnung: {html.escape(str(source))}</div>"

def _stoffdaten_comparison_html(cards, open_by_default=True):
    if not any(_has_stoffdaten(c) for c in cards):
        return ""
    labels=[
        ("Aggregatzustand", "aggregatzustand_raumtemperatur"),
        ("Schmelzpunkt / °C", "schmelzpunkt_C"),
        ("Siedepunkt / °C", "siedepunkt_C"),
        ("Dichte / g cm⁻³", "dichte_g_cm3"),
        ("Wasserlöslichkeit", "wasserloeslichkeit"),
    ]
    rows=[]
    for label,key in labels:
        vals=[]
        has=False
        for c in cards:
            sd=c.get("stoffdaten") or {}
            val=sd.get(key)
            if val not in (None, "", []): has=True
            vals.append("—" if val in (None, "", []) else html.escape(str(val)))
        if has:
            rows.append("<tr><td style='padding:4px 8px;border-bottom:1px solid #eee;'><b>"+html.escape(label)+"</b></td>"+"".join(f"<td style='padding:4px 8px;border-bottom:1px solid #eee;'>{v}</td>" for v in vals)+"</tr>")
    if not rows:
        return ""
    header="<tr><th style='text-align:left;padding:4px 8px;'>Stoffdatum</th>"+"".join(f"<th style='text-align:left;padding:4px 8px;'>{html.escape(c.get('name',''))}</th>" for c in cards)+"</tr>"
    open_attr=" open" if open_by_default else ""
    return f"""
    <details{open_attr} style='margin:10px 0;'>
      <summary><b>Stoffdaten / experimenteller Bezug</b></summary>
      <div style='font-size:0.9em;color:#333;padding:6px 0;'>
        <table style='border-collapse:collapse;width:100%;'>{header}{''.join(rows)}</table>
        <div style='font-size:0.78em;color:#666;margin-top:5px;'>Kuratierte Unterrichtswerte; Werte können je nach Quelle, Temperatur, Druck und Stoffform leicht abweichen. Sie dienen hier dem Struktur–Eigenschaft-Vergleich.</div>
      </div>
    </details>
    """

def _has_reactivity(card):
    r = card.get("reaktivitaet") or {}
    return any(r.get(k) not in (None, "", []) for k in ["hinweis", "warum", "reaktionsidee"])

def _reactivity_box_html(card, open_by_default=False):
    r = card.get("reaktivitaet") or {}
    if not _has_reactivity(card):
        return ""
    open_attr = " open" if open_by_default else ""
    return f"""
    <details{open_attr} style='margin:10px 0; border:1px solid #e5e5e5; border-radius:8px; padding:7px; background:#fbfbff;'>
      <summary style='cursor:pointer;'><b>Reaktivitäts-Hinweis anzeigen</b></summary>
      <div style='font-size:0.92em; line-height:1.35; padding-top:6px;'>
        <b>Hinweis:</b> {html.escape(_safe_text(r.get('hinweis')))}<br>
        <b>Warum?</b> {html.escape(_safe_text(r.get('warum')))}<br>
        <b>Reaktionsidee:</b> {html.escape(_safe_text(r.get('reaktionsidee')))}
        {('<br><span style="font-size:0.85em;color:#666;"><b>Modellgrenze:</b> '+html.escape(str(r.get('modellgrenze')))+' </span>') if r.get('modellgrenze') else ''}
      </div>
    </details>
    """

def _reactivity_comparison_html(cards):
    if not any(_has_reactivity(c) for c in cards):
        return ""
    rows=[]
    for field,label in [("hinweis","Hinweis"),("warum","Warum?"),("reaktionsidee","Reaktionsidee")]:
        vals=[]; has=False
        for c in cards:
            val=(c.get("reaktivitaet") or {}).get(field)
            if val not in (None,"",[]): has=True
            vals.append("—" if val in (None,"",[]) else html.escape(str(val)))
        if has:
            rows.append("<tr><td style='padding:4px 8px;border-bottom:1px solid #eee;'><b>"+html.escape(label)+"</b></td>"+"".join(f"<td style='padding:4px 8px;border-bottom:1px solid #eee;'>{v}</td>" for v in vals)+"</tr>")
    header="<tr><th style='text-align:left;padding:4px 8px;'>Reaktivität</th>"+"".join(f"<th style='text-align:left;padding:4px 8px;'>{html.escape(c.get('name',''))}</th>" for c in cards)+"</tr>"
    return f"""
    <details style='margin:10px 0;'>
      <summary><b>Reaktivitäts-Hinweise vergleichen</b></summary>
      <div style='font-size:0.9em;color:#333;padding:6px 0;'>
        <table style='border-collapse:collapse;width:100%;'>{header}{''.join(rows)}</table>
        <div style='font-size:0.78em;color:#666;margin-top:5px;'>Qualitative Orientierung anhand funktioneller Gruppen; keine vollständige Reaktionsvorhersage.</div>
      </div>
    </details>
    """

def display_structure_panel(mol, mol3d, props=None, groups=None, style="Kugel-Stab"):
    """Kompakte 2D-/3D-/Eigenschaften-Ansicht in einer Zeile, soweit Bildschirmbreite vorhanden ist."""
    png = mol_png_data_uri(mol, 300, 230)
    iframe = mol3d_iframe_html(mol3d, width=460, height=330, style=style, mode=render_mode_dd.value)
    props_html = _props_table_html(props or {}) if props else ""
    groups_html = ", ".join(groups or []) if groups else "keine aus der einfachen Liste erkannt"
    display(HTML(f"""
    <div style="display:flex; flex-wrap:wrap; gap:14px; align-items:flex-start; margin-top:8px;">
      <div style="flex:0 0 315px; max-width:100%;">
        <h3 style="margin:0 0 6px 0;">2D-Struktur</h3>
        <div style="border:1px solid #ddd; border-radius:8px; background:white; padding:8px; text-align:center;">
          <img src="{png}" style="max-width:100%; height:auto;">
        </div>
      </div>
      <div style="flex:0 0 500px; max-width:100%;">
        <h3 style="margin:0 0 6px 0;">3D-Modell</h3>
        {iframe}
        {_surface_legend_html(style)}
      </div>
      <div style="flex:1 1 330px; min-width:300px; max-width:520px;">
        <h3 style="margin:0 0 6px 0;">Eigenschaften</h3>
        <div style="border:1px solid #ddd; border-radius:8px; background:white; padding:8px;">
          {props_html}
          {_polarity_panel_html(mol, props) if props else ""}
          {_descriptor_note_html() if props else ""}
        </div>
        <h3 style="margin:12px 0 6px 0;">Funktionelle Gruppen</h3>
        <div style="border:1px solid #ddd; border-radius:8px; background:white; padding:8px;">{html.escape(groups_html)}</div>
      </div>
    </div>
    """))


def _structure_card_html(card, style="Kugel-Stab", width3d=330, height3d=255):
    """Kompakte HTML-Karte mit 2D- und 3D-Struktur für Vergleichsansichten."""
    mol = mol_from_smiles(card["smiles"])
    mol3d = make_3d(mol)
    png = mol_png_data_uri(mol, 230, 175)
    iframe = mol3d_iframe_html(mol3d, width=width3d, height=height3d, style=style, mode=render_mode_dd.value)
    return f"""
    <div style="flex:1 1 430px; min-width:390px; max-width:520px; border:1px solid #ddd; border-radius:10px; padding:10px; background:#fff;">
      <h3 style="margin-top:0;">{html.escape(card['name'])}</h3>
      <p style="margin:4px 0 8px 0;"><b>SMILES:</b> <code>{html.escape(card['smiles'])}</code></p>
      <div style="display:flex; flex-wrap:wrap; gap:10px; align-items:flex-start;">
        <div style="flex:0 0 240px; text-align:center;">
          <b>2D</b><br>
          <img src="{png}" style="max-width:100%; height:auto;">
        </div>
        <div style="flex:1 1 340px; min-width:320px;">
          <b>3D</b><br>
          {iframe}
          {_surface_legend_html(style)}
        </div>
      </div>
    </div>
    """

def display_isomer_comparison(card_a, card_b, style="Kugel-Stab"):
    """Isomerenpaar nebeneinander mit Strukturen und Eigenschaften darstellen."""
    mol_a = mol_from_smiles(card_a["smiles"])
    mol_b = mol_from_smiles(card_b["smiles"])
    props_a = calc_properties(mol_a)
    props_b = calc_properties(mol_b)
    groups_a = detect_functional_groups(mol_a)
    groups_b = detect_functional_groups(mol_b)
    display(HTML("""
    <h3>Isomeren-Vergleich</h3>
    <p>Beide Moleküle werden direkt gegenübergestellt. Achte besonders darauf, ob die <b>Summenformel gleich</b> ist, aber funktionelle Gruppen, räumliche Anordnung und Eigenschaften unterschiedlich sein können.</p><p style='font-size:0.92em;color:#555'><b>Hinweis:</b> Manche berechneten Modell-Deskriptoren, z. B. logP oder TPSA, können bei bestimmten Isomeren gleich ausfallen. Das bedeutet nicht, dass die Stoffe identisch sind; Reaktivität, Dipolmoment, Siedepunkt oder biologische Wirkung können trotzdem deutlich verschieden sein.</p>
    """))
    display(HTML("<div style='display:flex; flex-wrap:wrap; gap:16px; align-items:flex-start;'>" +
                 _structure_card_html(card_a, style=style) +
                 _structure_card_html(card_b, style=style) +
                 "</div>"))
    keys = ["Summenformel", "Molmasse / g mol⁻¹", "H-Brücken-Donoren", "H-Brücken-Akzeptoren", "logP (Modellwert)", "TPSA / Å²", "rotierbare Bindungen", "Ringanzahl"]
    rows=[]
    for k in keys:
        rows.append({"Eigenschaft": k, card_a["name"]: props_a.get(k,""), card_b["name"]: props_b.get(k,"")})
    qa = qualitative_polarity_assessment(mol_a, props_a)
    qb = qualitative_polarity_assessment(mol_b, props_b)
    rows.append({"Eigenschaft":"Didaktischer Polaritätseindruck", card_a["name"]: qa["polarity_label"], card_b["name"]: qb["polarity_label"]})
    rows.append({"Eigenschaft":"Wasserlöslichkeits-Tendenz", card_a["name"]: qa["solubility_label"], card_b["name"]: qb["solubility_label"]})
    rows.append({"Eigenschaft":"erkannte funktionelle Gruppen", card_a["name"]:", ".join(groups_a) or "keine erkannt", card_b["name"]:", ".join(groups_b) or "keine erkannt"})
    display(pd.DataFrame(rows))
    display(HTML(_descriptor_note_html()))
    display(HTML(_stoffdaten_comparison_html([card_a, card_b], open_by_default=True)))
    display(HTML(_reactivity_comparison_html([card_a, card_b])))
    display(HTML("""
    <p><b>Arbeitsauftrag:</b> Formuliere in einem Satz, welche Strukturänderung den wichtigsten Unterschied zwischen den beiden Molekülen ausmacht. Überlege anschließend, welche Stoffeigenschaft dadurch besonders beeinflusst werden könnte.</p>
    """))

def _display_py3dmol(viewer, width=520, height=400):
    display(HTML(_py3dmol_iframe_html(viewer, width=width, height=height)))

def show_conf(mol, conf_id, title="Konformer", style="Kugel-Stab", width=360, height=300):
    mb = Chem.MolToMolBlock(mol, confId=conf_id)
    display(HTML(f"<b>{title}</b><br>" + molblock_3d_html(mb, width=width, height=height, style=style, mode=render_mode_dd.value)))

def conf_iframe_html(mol, conf_id, title="Konformer", style="Kugel-Stab", width=360, height=300):
    mb = Chem.MolToMolBlock(mol, confId=conf_id)
    frame = molblock_3d_html(mb, width=width, height=height, style=style, mode=render_mode_dd.value)
    return f"<div style='flex:1 1 380px; min-width:330px; max-width:520px;'><b>{html.escape(title)}</b><br>" + frame + "</div>"

def display_conformers_side_by_side(mol, conformer_list, style="Kugel-Stab", max_n=4):
    """Konformere horizontal nebeneinander als getrennte, unabhängig drehbare 3D-Fenster."""
    boxes=[]
    for i,(cid,e) in enumerate(conformer_list[:max_n]):
        boxes.append(conf_iframe_html(mol, cid, f"Konformer {i+1}, ΔE = {e} kcal/mol", style=style, width=340, height=280))
    display(HTML("<div style='display:flex; flex-wrap:wrap; gap:14px; align-items:flex-start;'>" + "".join(boxes) + "</div>"))

def interpret_properties(props):
    parts=[]
    logp=props.get("logP (Modellwert)",0)
    tpsa=props.get("TPSA / Å²",0)
    hbd=props.get("H-Brücken-Donoren",0)
    hba=props.get("H-Brücken-Akzeptoren",0)
    rot=props.get("rotierbare Bindungen",0)
    if hbd>0: parts.append("Das Molekül kann Wasserstoffbrücken spenden.")
    else: parts.append("Das Molekül kann keine Wasserstoffbrücken spenden.")
    if hba>0: parts.append("Es kann Wasserstoffbrücken aufnehmen.")
    if tpsa < 20: parts.append("Die berechnete polare Oberfläche ist klein; das spricht für eher unpolare Eigenschaften.")
    elif tpsa < 80: parts.append("Die berechnete polare Oberfläche ist mittelgroß; polare und unpolare Bereiche können nebeneinander vorkommen.")
    else: parts.append("Die berechnete polare Oberfläche ist groß; deutliche polare Bereiche sind zu erwarten.")
    if logp < 0: parts.append("Der logP-Wert spricht eher für hydrophile Eigenschaften.")
    elif logp < 3: parts.append("Der logP-Wert spricht für ein gemischtes Verhältnis zwischen hydrophilen und lipophilen Anteilen.")
    else: parts.append("Der logP-Wert spricht für ein eher lipophiles Molekül.")
    if rot >= 5: parts.append("Mehrere rotierbare Bindungen machen das Molekül vergleichsweise flexibel.")
    elif rot >= 1: parts.append("Das Molekül besitzt einige bewegliche Einfachbindungen.")
    else: parts.append("Das Molekül ist relativ starr.")
    return " ".join(parts)

def generate_conformers(smiles, n=60):
    """Einfache RDKit-Konformersuche mit Energiestufen-Filter.

    Viele mehrfach erzeugte Konformere landen nach der Optimierung im gleichen
    Minimum. Für den Unterricht ist es hilfreicher, unterschiedliche
    Energieniveaus zu zeigen statt fünfmal ΔE = 0.
    """
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    params.pruneRmsThresh = -1.0
    try:
        params.useRandomCoords = True
    except Exception:
        pass
    ids = list(AllChem.EmbedMultipleConfs(mol, numConfs=n, params=params))
    results=[]
    props = AllChem.MMFFGetMoleculeProperties(mol)
    for cid in ids:
        try:
            if props is not None:
                ff = AllChem.MMFFGetMoleculeForceField(mol, props, confId=cid)
            else:
                ff = AllChem.UFFGetMoleculeForceField(mol, confId=cid)
            ff.Minimize(maxIts=500)
            e = ff.CalcEnergy()
        except Exception:
            ff = AllChem.UFFGetMoleculeForceField(mol, confId=cid)
            ff.Minimize(maxIts=500)
            e = ff.CalcEnergy()
        results.append((cid,e))
    if not results:
        return mol, [], []
    mine=min(e for _,e in results)
    rel=[(cid, e-mine) for cid,e in results]
    rel.sort(key=lambda x:x[1])

    # unterschiedliche Energieniveaus auswählen; sonst würden bei Butan z.B.
    # viele äquivalente anti-Konformere mit ΔE = 0 erscheinen.
    unique=[]
    for cid,e in rel:
        if all(abs(e - eu) > 0.08 for _, eu in unique):
            unique.append((cid,e))
        if len(unique) >= 6:
            break
    rel_round=[(cid, round(e,2)) for cid,e in rel]
    unique_round=[(cid, round(e,2)) for cid,e in unique]
    return mol, unique_round, rel_round

def card_for(module, name):
    for c in MOLECULES[module]:
        if c["name"] == name:
            return c
    return None

def _stoffdaten_lines(card, prefix=""):
    sd = card.get("stoffdaten") or {}
    lines=[]
    fields=[("Aggregatzustand", "aggregatzustand_raumtemperatur"),("Schmelzpunkt / °C", "schmelzpunkt_C"),("Siedepunkt / °C", "siedepunkt_C"),("Dichte / g cm⁻³", "dichte_g_cm3"),("Wasserlöslichkeit", "wasserloeslichkeit")]
    for label,key in fields:
        val=sd.get(key)
        if val not in (None,"",[]):
            lines.append(f"{prefix}- {label}: {val}")
    return lines

def report_text(rep):
    """Erzeugt einen einfachen kopierbaren SchülerInnen-Protokolltext."""
    lines=[]
    lines.append("MOL_LAB DIDACTIC – SchülerInnen-Protokoll")
    lines.append("="*44)
    lines.append("")
    rtype=rep.get("type", "molecule")
    if rtype == "molecule":
        card=rep.get("card", {})
        lines.append(f"Molekül: {rep.get('name','')}")
        lines.append(f"SMILES: {rep.get('smiles','')}")
        lines.append("")
        lines.append("Leitfrage:")
        lines.append(rep.get("leitfrage", ""))
        lines.append("")
        lines.append("Wichtige Modellwerte:")
        for k,v in rep.get("props",{}).items():
            lines.append(f"- {k}: {v}")
        lines.append("- Funktionelle Gruppen: " + (", ".join(rep.get('groups',[])) or "keine aus der einfachen Liste erkannt"))
        if rep.get('assessment'):
            lines.append("- Didaktischer Polaritätseindruck: " + rep['assessment'].get('polarity_label',''))
            lines.append("- Wasserlöslichkeits-Tendenz: " + rep['assessment'].get('solubility_label',''))
            lines.append("- Kurzbegründung: " + rep['assessment'].get('reason',''))
        sd_lines=_stoffdaten_lines(card)
        if sd_lines:
            lines.append("")
            lines.append("Stoffdaten / experimenteller Bezug:")
            lines.extend(sd_lines)
        if _has_reactivity(card):
            r=card.get("reaktivitaet") or {}
            lines.append("")
            lines.append("Reaktivitäts-Hinweis:")
            lines.append("- Hinweis: " + _safe_text(r.get("hinweis")))
            lines.append("- Warum? " + _safe_text(r.get("warum")))
            lines.append("- Reaktionsidee: " + _safe_text(r.get("reaktionsidee")))
        tasks=card.get("mini_aufgaben") or (card.get("didaktik") or {}).get("mini_aufgaben") or []
        if tasks:
            lines.append("")
            lines.append("Mini-Aufgaben:")
            for i,t in enumerate(tasks,1): lines.append(f"{i}. {t}")
        lines.append("")
        lines.append("Erklärung / Kernaussage:")
        lines.append(rep.get("erklaerung", ""))
        lines.append("")
        lines.append("Modellgrenze:")
        lines.append(rep.get("modellgrenze", ""))
    else:
        pair=rep.get("pair", {})
        cards=rep.get("cards", [])
        lines.append("Vergleich: " + rep.get("title", ""))
        lines.append("")
        if pair.get("fehlvorstellung"):
            lines.append("Typische Fehlvorstellung:")
            lines.append(pair.get("fehlvorstellung", ""))
            lines.append("")
        lines.append("Leitfrage:")
        lines.append(pair.get("leitfrage") or rep.get("leitfrage", ""))
        lines.append("")
        for card, props, groups, assessment in zip(cards, rep.get("props", []), rep.get("groups", []), rep.get("assessments", [])):
            lines.append(f"Molekül: {card.get('name','')}")
            lines.append(f"SMILES: {card.get('smiles','')}")
            for key in ["Summenformel","Molmasse / g mol⁻¹","H-Brücken-Donoren","H-Brücken-Akzeptoren","logP (Modellwert)","TPSA / Å²"]:
                if key in props: lines.append(f"- {key}: {props[key]}")
            lines.append("- Funktionelle Gruppen: " + (", ".join(groups) or "keine aus der einfachen Liste erkannt"))
            if assessment:
                lines.append("- Polaritätseindruck: " + assessment.get("polarity_label", ""))
                lines.append("- Wasserlöslichkeits-Tendenz: " + assessment.get("solubility_label", ""))
            sd_lines=_stoffdaten_lines(card)
            if sd_lines:
                lines.extend(sd_lines)
            lines.append("")
        tasks=pair.get("mini_aufgaben") or []
        if tasks:
            lines.append("Mini-Aufgaben:")
            for i,t in enumerate(tasks,1): lines.append(f"{i}. {t}")
            lines.append("")
        if pair.get("experiment") or pair.get("experimenteller_bezug"):
            lines.append("Experimenteller Bezug:")
            lines.append(pair.get("experiment") or pair.get("experimenteller_bezug"))
            lines.append("")
        if pair.get("kernaussage"):
            lines.append("Kernaussage:")
            lines.append(pair.get("kernaussage"))
            lines.append("")
        if pair.get("modellgrenze"):
            lines.append("Modellgrenze:")
            lines.append(pair.get("modellgrenze"))
    return "\n".join(lines)


# 3D-Rendering-Aufräumen: bewusst deaktiviert.
# Aggressives Entfernen von 3D-iframes kann in Colab leere 3D-Anzeigen verursachen.
def cleanup_3d_frames():
    return

# Oberfläche
module_dd = widgets.Dropdown(options=list(MOLECULES.keys()), description="Modul:", layout=widgets.Layout(width="520px"))
example_dd = widgets.Dropdown(description="Beispiel:", layout=widgets.Layout(width="520px"))
custom_smiles = widgets.Text(value="", placeholder="optional eigener SMILES-Code, z.B. CCO", description="SMILES:", layout=widgets.Layout(width="700px"))
use_custom = widgets.Checkbox(value=False, description="eigenen SMILES verwenden")
style_dd = widgets.Dropdown(
    options=["Kugel-Stab", "Stabmodell", "Kalottenmodell", "Didaktische Polaritätsoberfläche"],
    value="Kugel-Stab",
    description="3D:",
    layout=widgets.Layout(width="420px")
)

render_mode_dd = widgets.Dropdown(
    options=[
        ("3Dmol.js leicht (empfohlen)", "custom"),
        ("py3Dmol iFrame (Fallback)", "py3dmol")
    ],
    value="custom",
    description="3D-Engine:",
    layout=widgets.Layout(width="350px")
)
run_btn = widgets.Button(description="Molekül analysieren", button_style="success")
answer_radio = widgets.RadioButtons(options=[], description="Antwort:", layout=widgets.Layout(width="900px"))
check_btn = widgets.Button(description="Antwort prüfen", button_style="info")
explain_btn = widgets.Button(description="Erklärung anzeigen")
report_btn = widgets.Button(description="SchülerInnen-Protokolltext erzeugen", layout=widgets.Layout(width="300px"))
conf_btn = widgets.Button(description="Konformere berechnen", button_style="warning")
compare_btn = widgets.Button(description="Isomerenpaar vergleichen", button_style="primary")
polar_pair_dd = widgets.Dropdown(
    options=[("— bitte wählen —", None)] + [(k, k) for k in POLARITY_PAIRS.keys()],
    value=None,
    description="Vergleich:",
    layout=widgets.Layout(width="430px")
)
polar_compare_btn = widgets.Button(
    description="Polaritäts-/Löslichkeitsvergleich",
    button_style="primary",
    layout=widgets.Layout(width="300px")
)
reset3d_btn = widgets.Button(description="3D-Anzeige neu aufbauen", layout=widgets.Layout(width="230px"))
out = widgets.Output()
feedback_out = widgets.Output()
report_out = widgets.Output()
protocol_out = widgets.Output()


def refresh_examples(*args):
    module = module_dd.value
    names = [c["name"] for c in MOLECULES[module]]
    # In Colab hängen Dropdowns gelegentlich, wenn Optionen direkt ersetzt werden.
    # Daher erst leeren, dann neu setzen und den ersten Wert aktiv auswählen.
    example_dd.options = []
    example_dd.options = tuple(names)
    if names:
        example_dd.value = names[0]

    # Vergleichssteuerung bewusst nur dort anzeigen, wo sie didaktisch geführt ist.
    polar_pair_dd.layout.display = "" if module == "Polarität und Löslichkeit" else "none"
    polar_compare_btn.layout.display = "" if module == "Polarität und Löslichkeit" else "none"
    polar_pair_dd.value = None
    compare_btn.layout.display = "" if module == "Isomerie" else "none"
    out.clear_output(); feedback_out.clear_output(); report_out.clear_output(); protocol_out.clear_output()

refresh_examples()
module_dd.observe(refresh_examples, names="value")

def reset_pair_on_example_change(*args):
    # Wenn ein anderes Einzelbeispiel gewählt wird, soll kein altes Vergleichspaar 
    # gedanklich "mitwandern". Im Modul 1 muss bewusst neu gewählt werden.
    if module_dd.value == "Polarität und Löslichkeit":
        polar_pair_dd.value = None
    report_out.clear_output()

example_dd.observe(reset_pair_on_example_change, names="value")

def analyze(_=None):
    global current_report
    cleanup_3d_frames()
    with out:
        clear_output()
        gc.collect()
        module=module_dd.value
        if use_custom.value:
            smiles=custom_smiles.value.strip()
            if not smiles:
                print("Bitte einen SMILES-Code eingeben oder das Häkchen entfernen."); return
            card={"name":"Eigenes Molekül", "smiles":smiles, "leitfrage":"Welche Strukturmerkmale könnten die Stoffeigenschaften beeinflussen?", "beobachten":["Suche polare Gruppen.","Achte auf die Molekülgröße.","Vergleiche logP, TPSA und H-Brücken."], "frage":"Welche Aussage ist bei eigenen Molekülen besonders wichtig?", "antworten":["Die berechneten Werte sind Modellwerte und müssen fachlich geprüft werden.","SMILES-Eingaben sind immer experimentelle Messwerte.","Die Summenformel allein reicht immer aus.","Große Moleküle sind in dieser App immer zuverlässig berechnet."], "richtig":0, "feedback":"Richtig: Die App gibt Modellhinweise, keine endgültigen Messwerte.", "erklaerung":"Eigene SMILES-Eingaben sind für kleine bis mittelkleine Moleküle gedacht. Prüfe immer, ob die Struktur chemisch sinnvoll ist.", "modellgrenze":"Für große, geladene oder ungewöhnliche Moleküle ist diese einfache RDKit-Analyse nur eingeschränkt geeignet."}
        else:
            card=card_for(module, example_dd.value)
            smiles=card["smiles"]
        try:
            mol=mol_from_smiles(smiles)
        except Exception as e:
            print("Fehler:", e); return
        props=calc_properties(mol)
        groups=detect_functional_groups(mol)
        assessment=qualitative_polarity_assessment(mol, props)
        interp=interpret_properties(props)
        display(HTML(f"<h2>{card['name']}</h2><p><b>SMILES:</b> <code>{smiles}</code></p>"))
        display(HTML(f"<h3>Leitfrage</h3><p>{card.get('leitfrage','')}</p>"))
        display(HTML("<h3>Beobachte</h3><ul>" + "".join(f"<li>{x}</li>" for x in card.get('beobachten',[])) + "</ul>"))
        display_structure_panel(mol, make_3d(mol), props=props, groups=groups, style=style_dd.value)
        display(HTML("<h3>Erste Modell-Deutung</h3><p>" + interp + "</p>"))
        display(HTML(_reactivity_box_html(card)))
        answer_radio.options=card.get("antworten",[])
        answer_radio.value=answer_radio.options[0] if answer_radio.options else None
        display(HTML("<h3>Vermute</h3><p>" + card.get('frage','') + "</p>"))
        display(answer_radio, widgets.HBox([check_btn, explain_btn]))
        current_report={"type":"molecule", "name":card["name"], "smiles":smiles, "card":card, "props":props, "groups":groups, "assessment":assessment, "leitfrage":card.get("leitfrage",""), "interpretation":interp, "erklaerung":card.get("erklaerung",""), "modellgrenze":card.get("modellgrenze","")}
    feedback_out.clear_output(); report_out.clear_output(); protocol_out.clear_output()

def check_answer(_=None):
    with feedback_out:
        clear_output()
        if not current_report: return
        card=current_report["card"]
        selected=answer_radio.value
        idx=list(answer_radio.options).index(selected)
        if idx == card.get("richtig",0):
            display(HTML("<p style='color:green'><b>Richtig.</b> " + card.get("feedback","") + "</p>"))
        else:
            right=card.get("antworten",[])[card.get("richtig",0)]
            display(HTML("<p style='color:#b00020'><b>Noch nicht ganz.</b> Die beste Antwort ist: <b>" + right + "</b></p><p>" + card.get("feedback","") + "</p>"))

def explain(_=None):
    with feedback_out:
        clear_output()
        if not current_report: return
        card=current_report.get("card", {})
        display(HTML("<h3>Erklärung</h3><p>" + current_report.get("erklaerung","") + "</p>" + _reactivity_box_html(card, open_by_default=True) + "<h4>Grenze des Modells</h4><p>" + current_report.get("modellgrenze","") + "</p>"))

def show_report(_=None):
    with protocol_out:
        clear_output()
        if not current_report:
            print("Bitte zuerst ein Molekül analysieren oder ein Vergleichspaar anzeigen.")
            return
        txt=report_text(current_report)
        display(HTML("<h3>SchülerInnen-Protokolltext</h3><p>Der Text kann kopiert und in ein Heft-/Lernplattform-Protokoll übernommen werden.</p>"))
        display(widgets.Textarea(value=txt, layout=widgets.Layout(width="100%", height="300px")))

def do_conformers(_=None):
    cleanup_3d_frames()
    with report_out:
        clear_output()
        if not current_report:
            print("Bitte zuerst ein Molekül analysieren."); return
        smiles=current_report["smiles"]
        mol0 = Chem.MolFromSmiles(smiles)
        if mol0 is None or mol0.GetNumHeavyAtoms() > 35:
            print("Konformer-Suche nur für kleine bis mittelgroße Moleküle empfohlen."); return
        display(HTML("<h3>Konformere: einfache RDKit/MMFF-Suche</h3><p>Relative Energien sind Modellwerte in kcal/mol. Mehrfach gefundene gleiche Minima werden zusammengefasst. Bei kleinen Molekülen wie Butan oder 1,2-Dichlorethan sind oft nur zwei echte Minima sinnvoll: anti und gauche. Ekliptische Formen sind Übergangslagen und verschwinden bei einer normalen Geometrieoptimierung.</p>"))
        mol, unique_rel, all_rel = generate_conformers(smiles, n=120)
        if not unique_rel:
            print("Keine Konformere erzeugt."); return
        display(pd.DataFrame([{
            "Energieniveau": i+1,
            "confId": cid,
            "relative Energie / kcal mol⁻¹": e
        } for i,(cid,e) in enumerate(unique_rel[:6])]))
        if len(unique_rel) <= 2:
            display(HTML("<p><b>Hinweis:</b> Dass hier nur wenige Konformere erscheinen, ist meist kein Fehler: Nach der Optimierung bleiben nur stabile Minima übrig. Für Butan sind anti und gauche die entscheidenden Minima; ekliptische Formen wären keine stabilen Konformere.</p>"))
        display(HTML("<h4>Vergleichsansicht</h4><p>Die Konformere werden nebeneinander in getrennten 3D-Fenstern angezeigt und können unabhängig voneinander gedreht werden.</p>"))
        display_conformers_side_by_side(mol, unique_rel, style=style_dd.value, max_n=4)



def _comparison_assessment_label(mol, props):
    a = qualitative_polarity_assessment(mol, props)
    pol_labels = ["sehr schwach", "schwach", "mittel", "stark", "sehr stark"]
    sol_labels = ["kaum wasserlöslich", "begrenzt", "mäßig", "gut", "sehr gut"]
    return pol_labels[a["polarity_index"]], sol_labels[a["solubility_index"]]


def display_polarity_comparison(pair_key, style="Kugel-Stab"):
    """Kuratierter Vergleich im Modul Polarität und Löslichkeit."""
    pair = POLARITY_PAIRS[pair_key]
    card_a = card_for("Polarität und Löslichkeit", pair["a"])
    card_b = card_for("Polarität und Löslichkeit", pair["b"])
    if not card_a or not card_b:
        print("Das Vergleichspaar wurde in der Moleküldatenbank nicht gefunden.")
        return

    mol_a = mol_from_smiles(card_a["smiles"]); mol_b = mol_from_smiles(card_b["smiles"])
    props_a = calc_properties(mol_a); props_b = calc_properties(mol_b)
    groups_a = detect_functional_groups(mol_a); groups_b = detect_functional_groups(mol_b)
    pol_a, sol_a = _comparison_assessment_label(mol_a, props_a)
    pol_b, sol_b = _comparison_assessment_label(mol_b, props_b)

    display(HTML(f"""
    <h3>Vergleichspaar: {html.escape(pair_key)}</h3>
    <div style='background:#fff7e6; border-left:4px solid #f0ad4e; padding:8px 10px; margin:8px 0;'>
      <b>Typische Fehlvorstellung:</b> {html.escape(pair.get('fehlvorstellung',''))}
    </div>
    <p><b>Leitfrage:</b> {html.escape(pair.get('leitfrage',''))}</p>
    """))

    display(HTML("<div style='display:flex; flex-wrap:wrap; gap:14px; align-items:flex-start;'>" +
                 _structure_card_html(card_a, style=style, width3d=310, height3d=235) +
                 _structure_card_html(card_b, style=style, width3d=310, height3d=235) +
                 "</div>"))

    rows=[]
    keys = ["Summenformel", "Molmasse / g mol⁻¹", "H-Brücken-Donoren", "H-Brücken-Akzeptoren", "logP (Modellwert)", "TPSA / Å²", "Rotierbare Bindungen"]
    for k in keys:
        rows.append({"Eigenschaft": k, card_a["name"]: props_a.get(k,""), card_b["name"]: props_b.get(k,"")})
    rows.append({"Eigenschaft": "Polaritätseindruck", card_a["name"]: pol_a, card_b["name"]: pol_b})
    rows.append({"Eigenschaft": "Wasserlöslichkeits-Tendenz", card_a["name"]: sol_a, card_b["name"]: sol_b})
    rows.append({"Eigenschaft": "erkannte Gruppen", card_a["name"]: ", ".join(groups_a) or "keine erkannt", card_b["name"]: ", ".join(groups_b) or "keine erkannt"})
    display(pd.DataFrame(rows))
    display(HTML(_descriptor_note_html()))
    display(HTML(_stoffdaten_comparison_html([card_a, card_b], open_by_default=True)))
    display(HTML(_reactivity_comparison_html([card_a, card_b])))

    display(HTML("<h4>Mini-Aufgaben</h4><ol>" + "".join(f"<li>{html.escape(x)}</li>" for x in pair.get('mini_aufgaben',[])) + "</ol>"))
    if pair.get("experiment"):
        display(HTML(f"""
        <details style='margin:8px 0;'>
          <summary><b>Experimenteller Bezug anzeigen</b></summary>
          <div style='font-size:0.92em; color:#333; padding:6px 0;'>{html.escape(pair.get('experiment',''))}</div>
        </details>
        """))
    display(HTML(f"""
    <details style='margin:8px 0;'>
      <summary><b>Lösungsvorschlag / Kernaussage anzeigen</b></summary>
      <div style='font-size:0.92em; color:#333; padding:6px 0;'><b>Kernaussage:</b> {html.escape(pair.get('kernaussage',''))}<br>
      <b>Modellgrenze:</b> {html.escape(pair.get('modellgrenze',''))}</div>
    </details>
    """))


def do_polarity_comparison(_=None):
    global current_report
    cleanup_3d_frames()
    with report_out:
        clear_output()
        if module_dd.value != "Polarität und Löslichkeit":
            print("Der Polaritäts-/Löslichkeitsvergleich ist nur im Modul Polarität und Löslichkeit aktiv.")
            return
        key = polar_pair_dd.value
        if not key:
            print("Bitte ein Vergleichspaar wählen.")
            return
        display_polarity_comparison(key, style=style_dd.value)
        pair=POLARITY_PAIRS[key]
        card_a=card_for("Polarität und Löslichkeit", pair["a"]); card_b=card_for("Polarität und Löslichkeit", pair["b"])
        mol_a=mol_from_smiles(card_a["smiles"]); mol_b=mol_from_smiles(card_b["smiles"])
        props_a=calc_properties(mol_a); props_b=calc_properties(mol_b)
        current_report={"type":"polarity_pair", "title":key, "pair":pair, "cards":[card_a, card_b], "props":[props_a, props_b], "groups":[detect_functional_groups(mol_a), detect_functional_groups(mol_b)], "assessments":[qualitative_polarity_assessment(mol_a, props_a), qualitative_polarity_assessment(mol_b, props_b)]}

def do_isomer_comparison(_=None):
    global current_report
    cleanup_3d_frames()
    with report_out:
        clear_output()
        if module_dd.value != "Isomerie":
            print("Der Vergleich ist nur im Modul Isomerie aktiv.")
            return
        if use_custom.value:
            print("Für eigene SMILES ist noch kein automatischer Paarvergleich definiert.")
            return
        name_a = example_dd.value
        name_b = ISOMER_PAIRS.get(name_a)
        if not name_b:
            print("Für dieses Beispiel ist noch kein Isomerenpaar hinterlegt.")
            return
        card_a = card_for("Isomerie", name_a)
        card_b = card_for("Isomerie", name_b)
        if not card_a or not card_b:
            print("Das hinterlegte Vergleichspaar wurde nicht gefunden.")
            return
        display_isomer_comparison(card_a, card_b, style=style_dd.value)
        mol_a=mol_from_smiles(card_a["smiles"]); mol_b=mol_from_smiles(card_b["smiles"])
        props_a=calc_properties(mol_a); props_b=calc_properties(mol_b)
        current_report={"type":"isomer_pair", "title":card_a["name"]+" vs. "+card_b["name"], "pair":{"leitfrage":"Wie beeinflusst die andere Anordnung der Atome die Eigenschaften?", "mini_aufgaben":["Vergleiche Summenformel und Struktur.", "Suche unterschiedliche funktionelle Gruppen oder räumliche Anordnungen.", "Formuliere eine Struktur–Eigenschaft-Aussage."], "kernaussage":"Isomere können trotz gleicher Summenformel deutlich unterschiedliche Eigenschaften und Reaktivitäten besitzen.", "modellgrenze":"Modellwerte zeigen Trends, ersetzen aber keine experimentellen Stoffdaten."}, "cards":[card_a, card_b], "props":[props_a, props_b], "groups":[detect_functional_groups(mol_a), detect_functional_groups(mol_b)], "assessments":[qualitative_polarity_assessment(mol_a, props_a), qualitative_polarity_assessment(mol_b, props_b)]}

def reset_3d(_=None):
    """Sicherheitsknopf: 3D-iframes entfernen und letzte Analyse neu aufbauen."""
    cleanup_3d_frames()
    feedback_out.clear_output(); report_out.clear_output(); protocol_out.clear_output()
    gc.collect()
    analyze()

run_btn.on_click(analyze)
check_btn.on_click(check_answer)
explain_btn.on_click(explain)
report_btn.on_click(show_report)
conf_btn.on_click(do_conformers)
compare_btn.on_click(do_isomer_comparison)
polar_compare_btn.on_click(do_polarity_comparison)
reset3d_btn.on_click(reset_3d)

display(HTML('<h1>MOL_LAB DIDACTIC <span style="font-size:0.55em;color:#666">v0.4.0</span></h1><p>Wähle ein Modul und ein Beispiel oder gib einen eigenen SMILES-Code ein.</p>'))
controls = widgets.VBox([
    widgets.HTML("<b>1. Lernmodul und Beispiel auswählen</b>"),
    module_dd,
    example_dd,
    widgets.HTML("<b>2. 3D-Darstellung</b>"),
    style_dd,
    widgets.HBox([render_mode_dd, reset3d_btn]),
    widgets.HTML("<b>3. Optional: eigenes Molekül</b>"),
    use_custom,
    custom_smiles,
    widgets.HTML("<b>4. Analyse und geführte Vergleiche</b>"),
    widgets.HBox([polar_pair_dd, polar_compare_btn]),
    widgets.HBox([run_btn, conf_btn, compare_btn, report_btn]),
    widgets.HTML("<small>Hinweis: Die Vergleichspaare sind bewusst kuratiert. Freie SMILES-Eingaben sind zum Erkunden gedacht, ersetzen aber keine fachliche Prüfung.</small>")
])
display(controls, out, feedback_out, report_out, protocol_out)


## 3. Hinweise für weitere Versionen

In v0.4.0 neu bzw. sichtbar aktiviert:
- einfacher SchülerInnen-Protokolltext über eine einzige Schaltfläche
- Reaktivitäts-Hinweise als aufklappbare Dreischritt-Erklärung: Hinweis – Warum? – Reaktionsidee
- Stoffdaten/experimenteller Bezug in Molekülvergleichen und Isomerenvergleichen
- JSON-Struktur bleibt Grundlage für spätere alternative Niveaustufen und einen möglichen Editor
- 3D-Rendering bleibt bewusst auf der stabilen v0.3.4-Logik
